# Regulon, pathway, meta-program-network, centrality and survival analyses (TNBC and glioblastoma)

Notebook behind the later Results sections of the manuscript: regulon activity classes (Figs 7a, 8a), pathway enrichment
(Figs 7b, 8b; Supplementary Fig. S5), meta-program sub-networks with bootstrap support (Figs 7c, 8c; Supplementary Figs S7-S8),
degree/betweenness outlier TFs (Figs 7d, 8d) and TCGA Kaplan-Meier survival analyses (Supplementary Fig. S9).
Outputs are stripped and the machine-specific paths of the original have been replaced by `DATA_ROOT` / `GENE_ORDER_FILE` (next cell).
The code is otherwise as it was run, so it is a working notebook rather than a polished pipeline: run it top to bottom per cancer type
by editing the `SAMPLE` / `specimen` / `datasets` variables (the TNBC and glioblastoma settings are both present, one commented out).

| Cells (numbered from the imports cell = 0, i.e. after the two header cells) | Section |
|---|---|
| 0-2 | Imports; save / restore the session namespace (`*_pathway_analysis_shelve.pkl`) so a long session can be resumed |
| 3-9 | Load one sample's clone-labelled h5ad, clone UMAP, CNA heatmaps, genes with differential copy number between clones (`DECNP_genes`) |
| 10-49 | Regulon activity: AUCell on CollecTRI regulons (decoupler), differential activity across clones and samples, three regulon classes (clone-variable / sample-specific / multi-sample), overlap with copy-number-altered genes; regulon clustermap |
| 50-76 | Pathway analysis: g:Profiler and MSigDB WikiPathways / KEGG / Hallmark enrichment of each regulon class; regulon-vs-pathway Jaccard; pathway clustermaps |
| 77-102 | Bootstrap and SCENIC analysis: per-clone CellOracle / SCENIC networks restricted to meta-program genes, annotated with bootstrap support |
| 103-118 | Centrality: degree / betweenness outlier TFs per meta-program across clone pairings; pathway heatmaps |
| 119-128 | Patient survival: TCGA (UCSC Xena) BRCA-TNBC and GBM, Kaplan-Meier / log-rank for regulon and meta-program gene sets |

**Inputs this notebook expects** (none are shipped): for each TNBC / glioblastoma sample analysed, a clone-labelled h5ad
`<dataset>/<sample>_clone(_correct).h5ad` with `obs['inferred_clone']` (clone labels, `N` = normal) and `uns['cnp']` (copy-number matrix); the per-clone CellOracle and SCENIC edge lists
`bootstrap_celloracle/<dataset>/<sample>/CL<clone>-{whole|1..100}-{celloracle|scenic}-net.tsv` (written by `scripts/03_grn_inference/run_scenic_from_celloracle.py --export-celloracle`);
`Glio/Glio_meta_modules.csv`, `CollecTRI_regulons.csv`, `msig_human_WP_H_KEGG_new.csv`, `WP_KEGG_pathways.csv`, `GBM_relevant_pathways.csv`, `GBM_top30_pathways.csv`,
`TNBC_survival_programs.csv`, `Glio_survival_genes.csv` (gene-set / pathway tables); and internet access for TCGA (Xena) and g:Profiler. Cells marked `DO NOT RUN` are superseded earlier versions.


In [ ]:
import os
# Edit these two lines. DATA_ROOT is the directory the original analysis called '.../data/3CA'.
DATA_ROOT = os.environ.get('ITH_DATA_ROOT', '/path/to/data/3CA')
GENE_ORDER_FILE = os.environ.get('ITH_GENE_ORDER', '/path/to/hg38_gencode_v27.txt')   # tab-separated gene order file (chrom order in column 2)
os.makedirs(DATA_ROOT + '/manuscript_figs', exist_ok=True)   # figure PDFs are written here


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import math
import decoupler as dc
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import xenaPython as xena
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import networkx as nx
from tqdm import tqdm
import pickle as pkl
from gprofiler import GProfiler
import seaborn as sns
import shelve
import pdb
import io

plt.rcParams['figure.dpi'] = 300

In [ ]:
user_vars = {name: val for name, val in globals().items() if not (name.startswith('__') or name.startswith('_')) and not callable(val) and not (isinstance(val, type(pkl)) or isinstance(val, io.BufferedWriter))}

# Save the dictionary to a file
#pkl.dump({k:v for k,v in user_vars.items() if not k.startswith(('_', 'In', 'Out', 'user_vars'))}, open(DATA_ROOT + '/TNBC_pathway_analysis_shelve.pkl', 'wb'))

pkl.dump({k:v for k,v in user_vars.items() if not k.startswith(('_', 'In', 'Out', 'f', 'user_vars'))}, open(DATA_ROOT + '/Glio_pathway_analysis_shelve.pkl', 'wb'))

In [ ]:
with open(DATA_ROOT + '/Glio_pathway_analysis_shelve.pkl', 'rb') as f:
    loaded_data = pkl.load(f)

# Optional: Update the global namespace with the loaded variables
globals().update(loaded_data)

In [ ]:
#SAMPLE = 'CID4495'
#specimen = 'BC'
#'MGH102', 'MGH105', 'MGH122', 'MGH125', 'MGH136', 'MGH152'
#'SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R'

#{'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}

SAMPLE = 'TNBC2'
specimen = 'Gao_2021'#'Glio/Data_Neftel2019_Brain/SmartSeq2'
SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'

gene_order = pd.read_csv(GENE_ORDER_FILE, names = range(5), sep='\t')
gene_order = gene_order.sort_values(2)[0].tolist()

In [ ]:
adata = sc.read_h5ad(SAMPLE_PATH)
adata_dna = sc.AnnData(adata.uns['cnp'])#sc.AnnData(adata.uns['CNA'])
adata_dna.obs_names = adata.obs_names.copy()
adata_dna.var_names = adata.var_names.copy()
adata_dna.obs = adata.obs.copy()
adata_dna.var = adata.var.copy()

clones = adata.obs.inferred_clone.unique()

print(adata.obs['inferred_clone'].unique())

for clone in clones:
    if clone=='N' or len(adata.obs.loc[adata.obs['inferred_clone']==clone]) < 20:
        adata = adata[adata.obs.loc[adata.obs['inferred_clone']!=clone].index].copy()
        
print(adata.obs['inferred_clone'].unique())

sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata=adata, color="inferred_clone")

In [ ]:
plt.hist(np.count_nonzero(adata.to_df(), axis=0), bins=20)
plt.xlabel('no. of cells')
plt.ylabel('no. of genes')
plt.title('Distribution of gene counts across cells')
plt.show()

In [ ]:
adata.obs.inferred_clone.value_counts()

In [ ]:
sc.pl.heatmap(adata_dna, groupby='inferred_clone', var_names=[x for x in gene_order if x in adata_dna.var_names], vcenter=0, cmap='RdBu_r')

adata_dna = adata_dna[adata.obs_names, adata.var_names]
adata_dna = adata_dna[adata_dna.obs.loc[adata_dna.obs['inferred_clone']!='N'].index].copy()
#adata_dna = adata_dna[adata_dna.obs.loc[adata_dna.obs['inferred_clone']!='2'].index].copy()

In [ ]:
dna_copy = adata_dna.copy()
sc.tl.rank_genes_groups(dna_copy, groupby="inferred_clone", method="wilcoxon")
sc.pl.rank_genes_groups_heatmap(dna_copy, n_genes=300, show_gene_labels=False, vcenter=0, cmap='RdBu_r')

In [ ]:
#{'MGH105':3, 'MGH102':2, 'MGH122':2, 'MGH125':2, 'MGH136':2, 'MGH152':3}
cnp_thresh = 2

DECNP_genes = {x:[] for x in adata.obs['inferred_clone'].unique()}
for i in range(len(dna_copy.uns['rank_genes_groups']['logfoldchanges'])):
    for j in range(len(dna_copy.uns['rank_genes_groups']['logfoldchanges'][i])):
        if (dna_copy.uns['rank_genes_groups']['logfoldchanges'][i][j] > cnp_thresh or dna_copy.uns['rank_genes_groups']['logfoldchanges'][i][j] < -1*cnp_thresh) and dna_copy.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
            DECNP_genes[str(j)].append(dna_copy.uns['rank_genes_groups']['names'][i][j])
            
sc.pl.heatmap(dna_copy, var_names=DECNP_genes, groupby='inferred_clone', vcenter=0, cmap='RdBu_r')

In [ ]:
#Regulon activity DE analysis
collectri = dc.op.collectri(organism="human")
dc.mt.aucell(data=adata, net=collectri)
regulon_score = dc.pp.get_obsm(adata=adata, key="score_aucell")
sc.tl.rank_genes_groups(adata=regulon_score, groupby="inferred_clone")
#regulon_score_df = regulon_score_df[(regulon_score_df["meanchange"] > 0) & (regulon_score_df["padj"] < 0.05)] #may have to use padj?

In [ ]:
'''
print(regulon_score_df.columns)
for i,row in regulon_score_df.iterrows():
    print(list(row))
''' 

sc.pl.rank_genes_groups_matrixplot(adata=regulon_score, groupby='inferred_clone', n_genes=10, min_logfoldchange=0.25)

regulon_score_df = []
for i in range(len(regulon_score.uns['rank_genes_groups']['logfoldchanges'])):
    for j in range(len(regulon_score.uns['rank_genes_groups']['logfoldchanges'][i])):
        if (regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j] > 0.25) and regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
            regulon_score_df.append({'group':str(j+1), 'name':regulon_score.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j]})

regulon_score_df = pd.DataFrame(regulon_score_df).sort_values(by='group')

In [ ]:
print(regulon_score_df.columns)
for i,row in regulon_score_df.iterrows():
    print(list(row))

In [ ]:
'''
n_markers = 20
regulon_markers = (
    regulon_score_df.groupby("group")
    .head(n_markers)
    .drop_duplicates("name")
    .groupby("group")["name"]
    .apply(lambda x: list(x))
    .to_dict()
)
regulon_markers
'''

regulon_markers = {x:[] for x in adata.obs['inferred_clone'].unique()}
threshold = 2000
cnp_gene_thresh = 1

for i,row in regulon_score_df.iterrows():
    print(row['group'], row['name'])
    targets = collectri.loc[collectri['source']==row['name']]['target'].tolist()
    selected_targets = []
    print(targets)
    for target in targets:
        if target in DECNP_genes[row['group']]:
            if DECNP_genes[row['group']].index(target) <= threshold:
                selected_targets.append(target)
    if len(selected_targets) >= cnp_gene_thresh:
        print(selected_targets)
        regulon_markers[row['group']].append(row['name'])
    else:
        print('not enough DECNP targets:', len(selected_targets))

In [ ]:
regulon_markers

In [ ]:
sc.pl.matrixplot(
    adata=regulon_score,
    var_names=regulon_markers,
    groupby="inferred_clone",
    dendrogram=True,
    #standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="viridis",
)

In [ ]:
#1) Regulons DA across samples 
#   1.1) Regulons active across all or most samples
#   1.2.1) Regulons with DECNP gene
#       1.2.1.1) DECNP gene is TF 
#       1.2.1.2) DECNP gene is target 
#   1.2.2) Regulons without DECNP gene
# 1.2) Regulons active across one or few samples
#2) Regulons DA across clone(s) 
#   2.1) Regulons with DECNP gene 
#       2.1.1) DECNP gene is TF 
#       2.1.2) DECNP gene is target 
#   2.2) Regulons without DECNP gene

#all_data = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
#           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}
def compute_thresholds(df, lfc_col='logfoldchange', pval_col='pvals_adj',
                       lfc_bins=20, pval_bins=10, target_tail=10):
    """
    Compute histogram-based thresholds.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing logfoldchange and pvals_adj columns
    lfc_col : str
        Column name for log fold change
    pval_col : str
        Column name for adjusted p-values
    lfc_bins : int
        Number of bins for logfoldchange histogram
    pval_bins : int
        Number of bins for pvals_adj histogram
    target_tail : int
        Desired number of observations remaining after threshold

    Returns
    -------
    dict
        {
            'logfoldchange_threshold': value,
            'pvals_adj_threshold': value
        }
    """

    # --- Histogram for logfoldchange ---
    lfc_vals = df[lfc_col].dropna().values
    counts, edges = np.histogram(lfc_vals, bins=lfc_bins)

    # cumulative counts from the right
    tail_counts = np.cumsum(counts[::-1])[::-1]

    # find first bin where remaining counts ~ target_tail
    idx = np.argmin(np.abs(tail_counts - target_tail))

    lfc_threshold = edges[idx + 1]

    # --- Histogram for pvals_adj ---
    pval_vals = df[pval_col].dropna().values
    counts_p, edges_p = np.histogram(pval_vals, bins=pval_bins)

    pval_threshold = edges_p[1]  # upper limit of first bin

    return lfc_threshold, pval_threshold


#all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 
#            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 
#            'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}
#all_data = {'Wu_2021':['CID4513']}
all_data = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}

DE_regulons = {'cnp':{}, 'non_cnp':{}}
sample_DECNP_genes = {}
clone_DECNP_genes = {}

cnp_thresh_all = {'MGH105':3, 'MGH102':2, 'MGH122':2, 'MGH125':2, 'MGH136':2, 'MGH152':3, 'SF10022':3, 'SF11979sn':3, 'SF4297':3, 'SF6996':3, 'SF9259R':3}
#cnp_thresh_all = {'CID4495':3, 'CID4513':1, 'CID4515':2, 'CID4523':1, 'CID44971':3, 'CID44991':3, 'Triple_negative_0126':3, 'Triple_negative_BRCA1_0131':3, 'Triple_negative_BRCA1_0554':3, 'TNBC1':3, 'TNBC2':3, 'TNBC3':2}

for specimen, SAMPLES in all_data.items():
    for SAMPLE in SAMPLES:
        print('sample',SAMPLE)
        DE_regulons['cnp'][SAMPLE] = {}
        DE_regulons['non_cnp'][SAMPLE] = {}
        sample_DECNP_genes[SAMPLE] = {}
        #specimen = 'Glio/Data_Neftel2019_Brain/SmartSeq2'
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone.h5ad'
        else:
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'

        gene_order = pd.read_csv(GENE_ORDER_FILE, names = range(5), sep='\t')
        gene_order = gene_order.sort_values(2)[0].tolist()

        adata = sc.read_h5ad(SAMPLE_PATH)
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            adata_dna = sc.AnnData(adata.uns['CNA'])
        else:
            adata_dna = sc.AnnData(adata.uns['cnp'])
        adata_dna.obs_names = adata.obs_names.copy()
        adata_dna.var_names = adata.var_names.copy()
        adata_dna.obs = adata.obs.copy()
        adata_dna.var = adata.var.copy()

        clones = adata.obs.inferred_clone.unique()
        for clone in clones:
            if clone=='N' or (specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2' and clone=='0') or (specimen=='Wang_2019' and clone=='2') or len(adata.obs.loc[adata.obs['inferred_clone']==clone]) < 20:
                adata = adata[adata.obs.loc[adata.obs['inferred_clone']!=clone].index].copy()
                
        print(adata.obs['inferred_clone'].unique())
        
        sc.pp.filter_cells(adata, min_genes=100)
        sc.pp.filter_genes(adata, min_cells=3)
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        sc.tl.pca(adata)
        sc.pp.neighbors(adata)
        sc.tl.umap(adata)
        
        adata_dna = adata_dna[adata.obs_names, adata.var_names]
        adata_dna = adata_dna[adata_dna.obs.loc[adata_dna.obs['inferred_clone']!='N'].index].copy()
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            adata_dna = adata_dna[adata_dna.obs.loc[adata_dna.obs['inferred_clone']!='0'].index].copy()
        elif specimen=='Wang_2019':
            adata_dna = adata_dna[adata_dna.obs.loc[adata_dna.obs['inferred_clone']!='2'].index].copy()
        dna_copy = adata_dna.copy()
        sc.tl.rank_genes_groups(dna_copy, groupby="inferred_clone", method="wilcoxon")
        
        cnp_thresh = cnp_thresh_all[SAMPLE]

        DECNP_genes = {x:[] for x in adata.obs['inferred_clone'].unique()}
        for i in range(len(dna_copy.uns['rank_genes_groups']['logfoldchanges'])):
            for j in range(len(dna_copy.uns['rank_genes_groups']['logfoldchanges'][i])):
                if (dna_copy.uns['rank_genes_groups']['logfoldchanges'][i][j] > cnp_thresh or dna_copy.uns['rank_genes_groups']['logfoldchanges'][i][j] < -1*cnp_thresh) and dna_copy.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
                    DECNP_genes[dna_copy.uns['rank_genes_groups']['names'].dtype.names[j]].append(dna_copy.uns['rank_genes_groups']['names'][i][j])
        
        sample_DECNP_genes[SAMPLE] = [item for sublist in DECNP_genes.values() for item in sublist]
        clone_DECNP_genes[SAMPLE] = DECNP_genes.copy()
        
        continue
        #Regulon activity DE analysis
        collectri = pd.read_csv(DATA_ROOT + '/CollecTRI_regulons.csv')
        dc.mt.aucell(data=adata, net=collectri)
        regulon_score = dc.pp.get_obsm(adata=adata, key="score_aucell")
        sc.tl.rank_genes_groups(adata=regulon_score, groupby="inferred_clone")
        sc.tl.filter_rank_genes_groups(adata=regulon_score, min_in_group_fraction=0.25, max_out_group_fraction=0.2, min_fold_change=0)
        
        ##Consider only top n DE regulons
        regulon_score_df = []
        for i in range(len(regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'])):
            for j in range(len(regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'][i])):
                if (regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'][i][j] > 0) and regulon_score.uns['rank_genes_groups_filtered']['pvals_adj'][i][j] < 0.05 and regulon_score.uns['rank_genes_groups_filtered']['names'][i][j]==regulon_score.uns['rank_genes_groups_filtered']['names'][i][j]:
                    regulon_score_df.append({'group':regulon_score.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j]})

        if len(regulon_score_df) > 0:
            regulon_score_df = pd.DataFrame(regulon_score_df)
            lfc_threshold, pval_threshold = compute_thresholds(regulon_score_df, target_tail=15)
            print('Set threshods to ',lfc_threshold,pval_threshold)
            
            regulon_score_df = []
            for i in range(len(regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'])):
                for j in range(len(regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'][i])):
                    if (regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'][i][j] > lfc_threshold) and regulon_score.uns['rank_genes_groups_filtered']['pvals_adj'][i][j] < pval_threshold and regulon_score.uns['rank_genes_groups_filtered']['names'][i][j]==regulon_score.uns['rank_genes_groups_filtered']['names'][i][j]:
                        regulon_score_df.append({'group':regulon_score.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j]})        
        
            regulon_score_df = pd.DataFrame(regulon_score_df).sort_values(by='group')
            regulon_markers = {x:[] for x in adata.obs['inferred_clone'].unique()}
            threshold = 2000
            cnp_gene_thresh = 1
            select_top_n = 20
            
            de_count = {x:0 for x in adata.obs['inferred_clone'].unique()}
            decnp_count = {x:0 for x in adata.obs['inferred_clone'].unique()}

            for i,row in regulon_score_df.iterrows():
                targets = collectri.loc[collectri['source']==row['name']]['target'].tolist()
                selected_targets = []
                for target in list(set(targets+[row['name']])):
                    if target in DECNP_genes[row['group']]:
                        if DECNP_genes[row['group']].index(target) <= threshold:
                            selected_targets.append(target)
                if len(selected_targets) >= cnp_gene_thresh:
                    if decnp_count[row['group']] < select_top_n:
                        if row['group'] not in DE_regulons['cnp'][SAMPLE].keys():
                            DE_regulons['cnp'][SAMPLE][row['group']] = {}
                        DE_regulons['cnp'][SAMPLE][row['group']][row['name']] = {'targets':targets, 'cnp_targets':selected_targets}
                        regulon_markers[row['group']].append(row['name'])
                        decnp_count[row['group']]+=1
                elif de_count[row['group']]  < select_top_n:
                    if row['group'] not in DE_regulons['non_cnp'][SAMPLE].keys():
                        DE_regulons['non_cnp'][SAMPLE][row['group']] = {}
                    DE_regulons['non_cnp'][SAMPLE][row['group']][row['name']] = {'targets':targets}
                    de_count[row['group']]+=1
        else:
            print('NO REGULONS FOUND FOR',SAMPLE)


In [ ]:
sc.pl.rank_genes_groups_heatmap(adata=regulon_score, key="rank_genes_groups_filtered", groupby='inferred_clone', n_genes=20, min_logfoldchange=0, standard_scale='var', show_gene_labels=True)

In [ ]:
temp_df = []
for i in range(len(regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'])):
    for j in range(len(regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'][i])):
        if (regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'][i][j] > 0) and regulon_score.uns['rank_genes_groups_filtered']['pvals_adj'][i][j] < 0.05 and regulon_score.uns['rank_genes_groups_filtered']['names'][i][j]==regulon_score.uns['rank_genes_groups_filtered']['names'][i][j]:
            temp_df.append({'group':regulon_score.uns['rank_genes_groups_filtered']['names'].dtype.names[j], 'name':regulon_score.uns['rank_genes_groups_filtered']['names'][i][j], 'logfoldchange':regulon_score.uns['rank_genes_groups_filtered']['logfoldchanges'][i][j], 'pval':regulon_score.uns['rank_genes_groups_filtered']['pvals'][i][j], 'pvals_adj':regulon_score.uns['rank_genes_groups_filtered']['pvals_adj'][i][j]})

temp_df = pd.DataFrame(temp_df).sort_values(by=['group', 'logfoldchange'], ascending=[True, False])#.to_csv(DATA_ROOT + '/temp_DE_sample_regulons.csv', index=False)

In [ ]:
print(temp_df.to_string())
#np.histogram(temp_df['logfoldchange'], bins=20)

In [ ]:
np.histogram(np.array(regulon_score.uns['rank_genes_groups']['logfoldchanges'].tolist()).flatten())

In [ ]:
for sample,cloneval in DE_regulons['cnp'].items():
    for clone, regulons in cloneval.items():
        for TF, targets in regulons.items():
            if TF in targets['cnp_targets']:
                print(sample,clone,TF, targets)

In [ ]:
DE_regulon_keys = {}
for cnp_type in DE_regulons.keys():
    for sample in DE_regulons[cnp_type].keys():
        for clone in DE_regulons[cnp_type][sample].keys():
            if sample+str(clone) not in DE_regulon_keys.keys():
                DE_regulon_keys[sample+str(clone)] = []
            DE_regulon_keys[sample+str(clone)] = list(set(DE_regulon_keys[sample+str(clone)]+list(DE_regulons[cnp_type][sample][clone].keys())))
            
[b.keys() for a in DE_regulons['cnp'].values() for b in a.values()]

In [ ]:
sum([len(b.keys()) for a in DE_regulons['cnp'].values() for b in a.values()])

In [ ]:
DE_regulons['cnp']['CID4513']

In [ ]:
#datasets = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
#           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}
datasets = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 
            'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}

all_DE_regulons = {'cnp':{}, 'non_cnp':{}}
all_adata = {}

for specimen, SAMPLES in datasets.items():
    for SAMPLE in SAMPLES:
        #specimen = 'Glio/Data_Neftel2019_Brain/SmartSeq2'
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone.h5ad'
        else:
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'
        adata = sc.read_h5ad(SAMPLE_PATH)
        clones = adata.obs.inferred_clone.unique()
        for clone in clones:
            if clone=='N' or (specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2' and clone=='0') or (specimen=='Wang_2019' and clone=='2') or len(adata.obs.loc[adata.obs['inferred_clone']==clone]) < 20:
                adata = adata[adata.obs.loc[adata.obs['inferred_clone']!=clone].index].copy()
        adata.obs['sample'] = SAMPLE
        adata.obs['sample_clone'] = SAMPLE+adata.obs['inferred_clone'].astype(str)
        all_adata[SAMPLE] = adata.copy()

adata_combined = sc.concat([x for x in all_adata.values()], join="outer")
adata_combined.obs['sample'] = adata_combined.obs['sample'].astype('category')
adata_combined.obs['sample_clone'] = adata_combined.obs['sample_clone'].astype('category')
sc.pp.filter_cells(adata_combined, min_genes=100)
sc.pp.filter_genes(adata_combined, min_cells=3)
sc.pp.normalize_total(adata_combined)
sc.pp.log1p(adata_combined)
sc.pp.highly_variable_genes(adata_combined, n_top_genes=2000)
sc.tl.pca(adata_combined)
sc.pp.neighbors(adata_combined)
sc.tl.umap(adata_combined)

#Regulon activity DE analysis
collectri = pd.read_csv(DATA_ROOT + '/CollecTRI_regulons.csv')
dc.mt.aucell(data=adata_combined, net=collectri)
regulon_score = dc.pp.get_obsm(adata=adata_combined, key="score_aucell")
sc.tl.rank_genes_groups(adata=regulon_score, groupby="sample")

regulon_score_df = []
for i in range(len(regulon_score.uns['rank_genes_groups']['logfoldchanges'])):
    for j in range(len(regulon_score.uns['rank_genes_groups']['logfoldchanges'][i])):
        if (regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j] > 0) and regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
            regulon_score_df.append({'group':regulon_score.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j]})

if len(regulon_score_df) > 0:
    regulon_score_df = pd.DataFrame(regulon_score_df)
    lfc_threshold, pval_threshold = compute_thresholds(regulon_score_df, target_tail=40)
    print('Set threshods to ',lfc_threshold,pval_threshold)
    
    regulon_score_df = []
    regulon_score_df = []
    for i in range(len(regulon_score.uns['rank_genes_groups']['logfoldchanges'])):
        for j in range(len(regulon_score.uns['rank_genes_groups']['logfoldchanges'][i])):
            if (regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j] > lfc_threshold) and regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j] < pval_threshold:
                regulon_score_df.append({'group':regulon_score.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j]})

    regulon_score_df = pd.DataFrame(regulon_score_df).sort_values(by='group')
    select_top_n = 20
    de_count = {x:0 for SAMPLES in datasets.values() for x in SAMPLES}
    decnp_count = {x:0 for SAMPLES in datasets.values() for x in SAMPLES}

    for i,row in regulon_score_df.iterrows():
        targets = collectri.loc[collectri['source']==row['name']]['target'].tolist()
        cnp_genes = list(set(targets+[row['name']]).intersection(set(sample_DECNP_genes[row['group']])))
        if len(cnp_genes) > 0:
            if decnp_count[row['group']] < select_top_n:
                if row['group'] not in all_DE_regulons['cnp'].keys():
                    all_DE_regulons['cnp'][row['group']] = {}
                all_DE_regulons['cnp'][row['group']][row['name']] = {'targets':targets, 'cnp_targets':cnp_genes}
                decnp_count[row['group']]+=1
        elif de_count[row['group']] < select_top_n:
            if row['group'] not in all_DE_regulons['non_cnp'].keys():
                all_DE_regulons['non_cnp'][row['group']] = {}
            all_DE_regulons['non_cnp'][row['group']][row['name']] = {'targets':targets}
            de_count[row['group']]+=1


In [ ]:
##sample and clone DE activity in the same method of inference

def compute_thresholds(df, lfc_col='logfoldchange', pval_col='pvals_adj',
                       lfc_bins=20, pval_bins=10, target_tail=10):
    """
    Compute histogram-based thresholds.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing logfoldchange and pvals_adj columns
    lfc_col : str
        Column name for log fold change
    pval_col : str
        Column name for adjusted p-values
    lfc_bins : int
        Number of bins for logfoldchange histogram
    pval_bins : int
        Number of bins for pvals_adj histogram
    target_tail : int
        Desired number of observations remaining after threshold

    Returns
    -------
    dict
        {
            'logfoldchange_threshold': value,
            'pvals_adj_threshold': value
        }
    """

    # --- Histogram for logfoldchange ---
    lfc_vals = df[lfc_col].dropna().values
    counts, edges = np.histogram(lfc_vals, bins=lfc_bins)

    # cumulative counts from the right
    tail_counts = np.cumsum(counts[::-1])[::-1]

    # find first bin where remaining counts ~ target_tail
    idx = np.argmin(np.abs(tail_counts - target_tail))

    lfc_threshold = edges[idx + 1]

    # --- Histogram for pvals_adj ---
    pval_vals = df[pval_col].dropna().values
    counts_p, edges_p = np.histogram(pval_vals, bins=pval_bins)

    pval_threshold = edges_p[1]  # upper limit of first bin

    return lfc_threshold, pval_threshold




datasets = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}
#datasets = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 
#            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 
#            'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}

all_DE_regulons = {'cnp':{}, 'non_cnp':{}}
all_non_DE_regulons = {'cnp':{}, 'non_cnp':{}}
DE_regulons = {'cnp':{}, 'non_cnp':{}}
all_adata = {}




for specimen, SAMPLES in datasets.items():
    for SAMPLE in SAMPLES:
        #specimen = 'Glio/Data_Neftel2019_Brain/SmartSeq2'
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone.h5ad'
        else:
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'
        adata = sc.read_h5ad(SAMPLE_PATH)
        clones = adata.obs.inferred_clone.unique()
        for clone in clones:#clone=='N' or 
            if (specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2' and clone=='0') or (specimen=='Wang_2019' and clone=='2') or len(adata.obs.loc[adata.obs['inferred_clone']==clone]) < 20:
                adata = adata[adata.obs.loc[adata.obs['inferred_clone']!=clone].index].copy()
        adata.obs['sample'] = SAMPLE
        adata.obs['sample_clone'] = SAMPLE+adata.obs['inferred_clone'].astype(str)
        adata.obs['sample_tumor'] = [x if x[-1]=='N' else x[:-1]+'T' for x in adata.obs['sample_clone']]
        all_adata[SAMPLE] = adata.copy()

adata_combined = sc.concat([x for x in all_adata.values()], join="outer")
adata_combined.obs['sample'] = adata_combined.obs['sample'].astype('category')
adata_combined.obs['sample_clone'] = adata_combined.obs['sample_clone'].astype('category')
sc.pp.filter_cells(adata_combined, min_genes=100)
sc.pp.filter_genes(adata_combined, min_cells=3)
sc.pp.normalize_total(adata_combined)
sc.pp.log1p(adata_combined)
sc.pp.highly_variable_genes(adata_combined, n_top_genes=2000, batch_key='sample')
sc.tl.pca(adata_combined)
sc.pp.neighbors(adata_combined)
sc.tl.umap(adata_combined)


#Regulon activity DE analysis
collectri = pd.read_csv(DATA_ROOT + '/CollecTRI_regulons.csv')
dc.mt.aucell(data=adata_combined, net=collectri)
regulon_score = dc.pp.get_obsm(adata=adata_combined, key="score_aucell")

#create sample and clone seprarately
regulon_score_sample = regulon_score.copy()
regulon_score_clone = regulon_score.copy()
sc.tl.rank_genes_groups(adata=regulon_score_sample, groupby="sample_tumor")
sc.tl.rank_genes_groups(adata=regulon_score_clone, groupby="sample_clone")

#sc.tl.filter_rank_genes_groups(adata=regulon_score_sample, groupby="sample_tumor", min_in_group_fraction=0.6, max_out_group_fraction=0.6, min_fold_change=0)



## sample level
#TNBC target_tail = 70, lfc_bins = 30
print('SAMPLE LEVEL')
regulon_score_df = []
for i in range(len(regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'])):
    for j in range(len(regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i])):
        if (regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i][j] > 0) and regulon_score_sample.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
            regulon_score_df.append({'group':regulon_score_sample.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score_sample.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score_sample.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score_sample.uns['rank_genes_groups']['pvals_adj'][i][j]})

if len(regulon_score_df) > 0:
    regulon_score_df = pd.DataFrame(regulon_score_df)
    lfc_threshold, pval_threshold = compute_thresholds(regulon_score_df, target_tail=140, lfc_bins=40)
    print('Set threshods to ',lfc_threshold,pval_threshold)
    
    regulon_score_df = []
    regulon_score_df = []
    for i in range(len(regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'])):
        for j in range(len(regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i])):
            if (regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i][j] > lfc_threshold) and regulon_score_sample.uns['rank_genes_groups']['pvals_adj'][i][j] < pval_threshold:
                regulon_score_df.append({'group':regulon_score_sample.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score_sample.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score_sample.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score_sample.uns['rank_genes_groups']['pvals_adj'][i][j]})

    regulon_score_df = pd.DataFrame(regulon_score_df).sort_values(by='group')
    select_top_n = 15
    de_count = {x:0 for x in regulon_score_df['group'].unique()}
    decnp_count = {x:0 for x in regulon_score_df['group'].unique()}

    for i,row in regulon_score_df.iterrows():
        if row['group'][-1]=='N':
            continue
        targets = collectri.loc[collectri['source']==row['name']]['target'].tolist()
        cnp_genes = list(set(targets+[row['name']]).intersection(set(sample_DECNP_genes[row['group'][0:-1]])))
        if len(cnp_genes) > 0:
            if decnp_count[row['group']] < select_top_n:
                if row['group'] not in all_DE_regulons['cnp'].keys():
                    all_DE_regulons['cnp'][row['group']] = {}
                all_DE_regulons['cnp'][row['group']][row['name']] = {'targets':targets, 'cnp_targets':cnp_genes}
                decnp_count[row['group']]+=1
        elif de_count[row['group']] < select_top_n:
            if row['group'] not in all_DE_regulons['non_cnp'].keys():
                all_DE_regulons['non_cnp'][row['group']] = {}
            all_DE_regulons['non_cnp'][row['group']][row['name']] = {'targets':targets}
            de_count[row['group']]+=1


## sample level multiple
#TNBC target_tail = 1, lfc_bins = 20, select_top_n=2
print('SAMPLE LEVEL')
regulon_score_df = []
for i in range(len(regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'])):
    for j in range(len(regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i])):
        if (regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i][j] > 0) and regulon_score_sample.uns['rank_genes_groups']['pvals_adj'][i][j] > 0.2:
            regulon_score_df.append({'group':regulon_score_sample.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score_sample.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score_sample.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score_sample.uns['rank_genes_groups']['pvals_adj'][i][j]})

if len(regulon_score_df) > 0:
    regulon_score_df = pd.DataFrame(regulon_score_df)
    lfc_threshold, pval_threshold = compute_thresholds(regulon_score_df, target_tail=1, lfc_bins=20)
    print('Set threshods to ',lfc_threshold,pval_threshold)
    
    regulon_score_df = []
    regulon_score_df = []
    for i in range(len(regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'])):
        for j in range(len(regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i])):
            if (regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i][j] > lfc_threshold) and regulon_score_sample.uns['rank_genes_groups']['pvals_adj'][i][j] < pval_threshold:
                regulon_score_df.append({'group':regulon_score_sample.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score_sample.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score_sample.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score_sample.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score_sample.uns['rank_genes_groups']['pvals_adj'][i][j]})

    regulon_score_df = pd.DataFrame(regulon_score_df).sort_values(by='group')
    select_top_n = 4
    de_count = {x:0 for x in regulon_score_df['group'].unique()}
    decnp_count = {x:0 for x in regulon_score_df['group'].unique()}

    for i,row in regulon_score_df.iterrows():
        if row['group'][-1]=='N':
            continue
        targets = collectri.loc[collectri['source']==row['name']]['target'].tolist()
        cnp_genes = list(set(targets+[row['name']]).intersection(set(sample_DECNP_genes[row['group'][0:-1]])))
        if len(cnp_genes) > 0:
            if decnp_count[row['group']] < select_top_n:
                if row['group'] not in all_non_DE_regulons['cnp'].keys():
                    all_non_DE_regulons['cnp'][row['group']] = {}
                all_non_DE_regulons['cnp'][row['group']][row['name']] = {'targets':targets, 'cnp_targets':cnp_genes}
                decnp_count[row['group']]+=1
        elif de_count[row['group']] < select_top_n:
            if row['group'] not in all_non_DE_regulons['non_cnp'].keys():
                all_non_DE_regulons['non_cnp'][row['group']] = {}
            all_non_DE_regulons['non_cnp'][row['group']][row['name']] = {'targets':targets}
            de_count[row['group']]+=1


## clone level
#TNBC target_tail = 150, lfc_bins = 40
print('CLONE LEVEL')
regulon_score_df = []
for i in range(len(regulon_score_clone.uns['rank_genes_groups']['logfoldchanges'])):
    for j in range(len(regulon_score_clone.uns['rank_genes_groups']['logfoldchanges'][i])):
        if (regulon_score_clone.uns['rank_genes_groups']['logfoldchanges'][i][j] > 0) and regulon_score_clone.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
            regulon_score_df.append({'group':regulon_score_clone.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score_clone.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score_clone.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score_clone.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score_clone.uns['rank_genes_groups']['pvals_adj'][i][j]})

if len(regulon_score_df) > 0:
    regulon_score_df = pd.DataFrame(regulon_score_df)
    lfc_threshold, pval_threshold = compute_thresholds(regulon_score_df, target_tail=200, lfc_bins=50)
    print('Set threshods to ',lfc_threshold,pval_threshold)
    
    regulon_score_df = []
    regulon_score_df = []
    for i in range(len(regulon_score_clone.uns['rank_genes_groups']['logfoldchanges'])):
        for j in range(len(regulon_score_clone.uns['rank_genes_groups']['logfoldchanges'][i])):
            if (regulon_score_clone.uns['rank_genes_groups']['logfoldchanges'][i][j] > lfc_threshold) and regulon_score_clone.uns['rank_genes_groups']['pvals_adj'][i][j] < pval_threshold:
                regulon_score_df.append({'group':regulon_score_clone.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score_clone.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score_clone.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score_clone.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score_clone.uns['rank_genes_groups']['pvals_adj'][i][j]})

    regulon_score_df = pd.DataFrame(regulon_score_df).sort_values(by='group')
    select_top_n = 8
    de_count = {x:0 for x in regulon_score_clone.obs['sample_clone']}
    decnp_count = {x:0 for x in regulon_score_clone.obs['sample_clone']}

    for i,row in regulon_score_df.iterrows():
        if row['group'][-1]=='N':
            continue
        targets = collectri.loc[collectri['source']==row['name']]['target'].tolist()
        cnp_genes = list(set(targets+[row['name']]).intersection(set(clone_DECNP_genes[row['group'][:-1]][row['group'][-1]])))
        if len(cnp_genes) > 0:
            if decnp_count[row['group']] < select_top_n:
                if row['group'] not in DE_regulons['cnp'].keys():
                    DE_regulons['cnp'][row['group']] = {}
                DE_regulons['cnp'][row['group']][row['name']] = {'targets':targets, 'cnp_targets':cnp_genes}
                decnp_count[row['group']]+=1
        elif de_count[row['group']] < select_top_n:
            if row['group'] not in DE_regulons['non_cnp'].keys():
                DE_regulons['non_cnp'][row['group']] = {}
            DE_regulons['non_cnp'][row['group']][row['name']] = {'targets':targets}
            de_count[row['group']]+=1


In [ ]:
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd
import re
from scipy import sparse
from itertools import combinations
from collections import defaultdict

def get_sample_group_pairs(adata, group_col):
    """
    Extract groups per sample and compute all pairwise group combinations.

    Parameters
    ----------
    adata : AnnData
    group_col : str
        Column in adata.obs containing values like 'CID44950'

    Returns
    -------
    dict
        {sample : [(group1, group2), (group1, group3), ...]}
    """

    sample_groups = defaultdict(set)

    for val in adata.obs[group_col]:
        sample = val[:-1]   # sample ID
        group = val[-1]     # group ID
        sample_groups[sample].add(group)

    pair_dict = {}

    for sample, groups in sample_groups.items():
        groups = sorted(groups)
        pair_dict[sample] = list(combinations(groups, 2))

    return pair_dict


def find_group_specific_genes(
    adata,
    group_col,
    high_thresh=0.8,
    low_thresh=0.2,
    min_samples=2
):  
    regulons = {}
    adata_df = adata.to_df().copy()
    adata_df = (adata_df > 0).astype(int)
    adata_df[group_col] = adata.obs[group_col].copy()
    adata_df = adata_df.groupby([group_col]).mean()
    group_pairings = get_sample_group_pairs(adata, group_col)
    for regulon in adata_df.columns:
        regulons[regulon] = []
        for sample, clone_pairs in group_pairings.items():
            if sample+'N' in adata_df.index:
                if adata_df.loc[sample+'N',regulon] >= 0.2:
                    continue
            for clone_pair in clone_pairs:
                if 'N' in clone_pair:
                    continue
                clone1 = sample+clone_pair[0]
                clone2 = sample+clone_pair[1]
                if (adata_df.loc[clone1,regulon] >= high_thresh and adata_df.loc[clone2,regulon] < low_thresh) or (adata_df.loc[clone2,regulon] >= high_thresh and adata_df.loc[clone1,regulon] < low_thresh):
                    regulons[regulon].append(sample)
                    break
        if len(regulons[regulon]) < min_samples:
            del regulons[regulon]
    return regulons, adata_df

def find_sample_specific_genes(
    adata,
    group_col,
    high_thresh=0.8,
    low_thresh=0.2,
    max_samples=2,
    min_samples=2
):  
    regulons = {}
    adata_df = adata.to_df().copy()
    adata_df = (adata_df > 0).astype(int)
    adata_df[group_col] = adata.obs[group_col].copy()
    adata_df = adata_df.groupby([group_col]).mean()
    adata_df_bin = adata_df.copy()
    adata_df_bin[adata_df_bin >= high_thresh] = 1
    adata_df_bin[adata_df_bin < low_thresh] = 3
    adata_df_bin[adata_df_bin < 1] = 2
    adata_df_bin[adata_df_bin == 3] = 0
    temp_df = adata_df_bin[~adata_df_bin.index.str.endswith('N')]
    for regulon in adata_df_bin.columns:
        regulons[regulon] = []
        if (temp_df.loc[temp_df[regulon]==1][regulon].sum() <= max_samples) and (temp_df.loc[temp_df[regulon]==2][regulon].sum() <= min_samples*2):
            for sample in temp_df.loc[temp_df[regulon]==1].index:
                if sample[:-1]+'N' in adata_df_bin.index:
                    if adata_df.loc[sample[:-1]+'N',regulon] < 0.2:
                        regulons[regulon].append(sample)
        if len(regulons[regulon]) == 0:
            del regulons[regulon]
        
    return regulons, adata_df_bin

#Glio
gene_hits, props = find_group_specific_genes(regulon_score, 'sample_clone',high_thresh=0.4,low_thresh=0.35,min_samples=2)
gene_hits, props = find_group_specific_genes(regulon_score, 'sample_clone',high_thresh=0.5,low_thresh=0.35,min_samples=2)
gene_sample_hits, sample_props = find_sample_specific_genes(regulon_score, 'sample_tumor',high_thresh=0.5,low_thresh=0.4,max_samples=1, min_samples=3)
gene_multi_sample_hits, multi_sample_props = find_sample_specific_genes(regulon_score, 'sample_tumor',high_thresh=0.4,low_thresh=0.2,max_samples=6, min_samples=3)

#TNBC
#gene_hits, props = find_group_specific_genes(regulon_score, 'sample_clone',high_thresh=0.4,low_thresh=0.35,min_samples=2)
#gene_sample_hits, sample_props = find_sample_specific_genes(regulon_score, 'sample_tumor',high_thresh=0.5,low_thresh=0.3,max_samples=1, min_samples=2)
#gene_multi_sample_hits, multi_sample_props = find_sample_specific_genes(regulon_score, 'sample_tumor',high_thresh=0.4,low_thresh=0.2,max_samples=6, min_samples=3)

In [ ]:
print(gene_hits)
print(gene_sample_hits)
print(gene_multi_sample_hits)

In [ ]:
for gene in gene_hits.keys():
    for sample in gene_hits[gene]:
        if gene in sample_DECNP_genes[sample[:]]:
            print(gene)

In [ ]:
mask_keep = ~regulon_score.obs['sample_clone'].isin([x for x in regulon_score.obs['sample_clone'].unique() if x[-1]=='N'])
sc.pl.heatmap(adata=regulon_score[mask_keep], var_names=list(gene_hits.keys()), groupby='sample_clone', standard_scale='var', figsize=(6,4))

In [ ]:
from scipy.stats import entropy

def diagonal_col_order(df):
    matrix = df.values
    row_indices = np.arange(matrix.shape[0])

    # Center of mass per column
    col_centers = (
        (matrix * row_indices[:, None]).sum(axis=0)
        / (matrix.sum(axis=0) + 1e-9)
    )

    # Distinctiveness per column
    def distinctiveness(col):
        p = col / (col.sum() + 1e-9)
        return 1 - entropy(p) / np.log(len(p) + 1e-9)

    distinct = np.array([distinctiveness(matrix[:, j]) for j in range(matrix.shape[1])])

    # Sort: primary = center of mass (asc), secondary = distinctiveness (desc)
    col_order = np.lexsort((-distinct, col_centers))

    return col_order


#sample_list = ['CID44971', 'CID4515', 'CID44991', 'CID4513', 'CID4495', 'Triple_negative_BRCA1_0554', 'CID4523', 'TNBC1', 'TNBC2', 'Triple_negative_BRCA1_0131', 'Triple_negative_0126']
#sample_list_color = {'CID4495':'#1F77B4', 'CID4513':'#FF7F0E', 'CID4515':'#279E68', 'CID4523':'#D62728', 'CID44971':'#AA40FC', 'CID44991':'#8C564B', 'TNBC1':'#E16EBE', 'TNBC2':'#B5BD61', 'TNBC3':'#17BECF', 'Triple_negative_0126':'#AEC7E8', 'Triple_negative_BRCA1_0131':'#FFBB78', 'Triple_negative_BRCA1_0554':'#98DF8A'}
#clone_level_order = ['FOXJ1', 'SIX1', 'MYBL1', 'NONO', 'GRHL2', 'PRDM2', 'MED1', 'PBRM1', 'ZKSCAN7', 'HEY1', 'HEY2']
#sample_level_order = ['DLX5', 'SOX10', 'TGIF1', 'ID4', 'NR2E1']
#multi_sample_level_order = ['FOXD3', 'HOXB13', 'MXI1', 'TCF7', 'BHLHE40', 'TRPS1', 'MAFG', 'LMX1B', 'SP7', 'KLF11', 'ZNF143', 'EHF', 'NFIX', 'NR1D1', 'ETV1', 'MAFB', 'TEAD4', 'PROX1', 'DLX2', 'BCL11A', 'PLAGL1', 'TBX3', 'PATZ1', 'POU2F3', 'NSD2', 'CTBP2', 'TBX5', 'POU3F1', 'E2F2', 'ZNF384', 'TCF21', 'FOXQ1', 'KLF13']
sample_list = ['MGH102', 'MGH105', 'MGH122', 'MGH125', 'MGH136', 'MGH152', 'SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']
sample_list_color = {'MGH102':'#1F77B4', 'MGH105':'#FF7F0E', 'MGH122':'#279E68', 'MGH125':'#D62728', 'MGH136':'#AA40FC', 'MGH152':'#8C564B', 'SF10022':'#E16EBE', 'SF11979sn':'#B5BD61', 'SF4297':'#17BECF', 'SF6996':'#AEC7E8', 'SF9259R':'#FFBB78'}
clone_level_order = ['ZNF354C', 'BCL11B', 'NR2F2', 'EMX2', 'HOXB7', 'KCNIP3', 'LMO2', 'NEUROD2']
sample_level_order = ['EBF1', 'PKNOX1', 'PEG3', 'CEBPZ']
multi_sample_level_order = ['SOX11', 'NONO', 'MSC', 'MNT', 'HMGB2', 'NEUROG1', 'MYT1', 'DLX2', 'FOXP1', 'HEY1', 'DLX1', 'WWTR1', 'SMARCC1', 'BCL11A', 'ZFPM2']


def reorder(df, samp_list, colors):
    sample_order = []
    sample_colors = []
    for sample in samp_list:
        temp_list = []
        temp_list2 = []
        for row in df.index:
            if sample in row:
                temp_list.append(row)
                temp_list2.append(colors[sample])
        temp_list.sort()
        sample_order+=temp_list
        sample_colors+=temp_list2
    print(sample_order)
    return sample_order, sample_colors

#sc.pl.matrixplot(adata=regulon_score, var_names=list(gene_hits.keys()), groupby='sample_clone', standard_scale='var', figsize=(6,4))
#sns.clustermap(props[~props.index.str.endswith('N')][gene_hits.keys()], figsize=(6,6), cmap='viridis', row_cluster=False, xticklabels=True, yticklabels=True)


#glio fig size: 4,7; 4,6; 10,4
#tnbc fig size: 6,6; 6,6; 10,6
mask_keep = ~regulon_score.obs['sample_clone'].isin([x for x in regulon_score.obs['sample_clone'].unique() if x[-1]=='N'])
mp = sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names=list(gene_hits.keys()), groupby='sample_clone', standard_scale='var', figsize=(6,4), return_fig=True)
sample_order, row_colors = reorder(mp.values_df, sample_list, sample_list_color)
col_order = diagonal_col_order(mp.values_df.loc[sample_order, :].copy())
col_order = clone_level_order#[mp.values_df.columns[x] for x in col_order.tolist()]
g = sns.clustermap(mp.values_df.loc[sample_order, col_order], row_colors=row_colors, figsize=(4,7), cmap='viridis', row_cluster=False, col_cluster=False, xticklabels=True, yticklabels=True)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90, horizontalalignment='center')
plt.show()


mp = sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names=list(gene_sample_hits.keys()), groupby='sample_tumor', standard_scale='var', figsize=(6,4), return_fig=True)
sample_order, row_colors = reorder(mp.values_df, sample_list, sample_list_color)
col_order = diagonal_col_order(mp.values_df.loc[sample_order, :].copy())
col_order = sample_level_order#[mp.values_df.columns[x] for x in col_order.tolist()]
g = sns.clustermap(mp.values_df.loc[sample_order,col_order], row_colors=row_colors, figsize=(4,6), cmap='viridis', row_cluster=False, col_cluster=False, xticklabels=True, yticklabels=True)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90, horizontalalignment='center')
plt.show()


mp = sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names=list(set(gene_multi_sample_hits.keys()).difference(set(gene_hits.keys()).union(set(gene_sample_hits.keys())))), groupby='sample_tumor', standard_scale='var', figsize=(6,4), return_fig=True)
sample_order, row_colors = reorder(mp.values_df, sample_list, sample_list_color)
col_order = diagonal_col_order(mp.values_df.loc[sample_order, :].copy())
col_order = multi_sample_level_order#[mp.values_df.columns[x] for x in col_order.tolist()]
print(col_order)
g = sns.clustermap(mp.values_df.loc[sample_order,col_order], row_colors=row_colors, figsize=(10,4), cmap='viridis', row_cluster=False, col_cluster=False, xticklabels=True, yticklabels=True)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90, horizontalalignment='center')
plt.show()

'''
mask_keep = ~regulon_score.obs['sample_clone'].isin([x for x in regulon_score.obs['sample_clone'].unique() if x[-1]=='N'])
mp = sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names=list(gene_hits.keys()), groupby='sample_clone', standard_scale='var', figsize=(6,4), return_fig=True)
sns.clustermap(mp.values_df, figsize=(6,6), cmap='viridis', row_cluster=False, xticklabels=True, yticklabels=True)
mp = sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names=list(gene_sample_hits.keys()), groupby='sample_tumor', standard_scale='var', figsize=(6,4), return_fig=True)
sns.clustermap(mp.values_df, figsize=(6,6), cmap='viridis', row_cluster=False, xticklabels=True, yticklabels=True)
mp = sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names=list(set(gene_multi_sample_hits.keys()).difference(set(gene_hits.keys()).union(set(gene_sample_hits.keys())))), groupby='sample_tumor', standard_scale='var', figsize=(6,4), return_fig=True)
sns.clustermap(mp.values_df, figsize=(10,6), cmap='viridis', row_cluster=False, xticklabels=True, yticklabels=True)
'''

In [ ]:
temp_genes = []
for tf in list(set([a for a in gene_hits.keys()])):
    temp_genes.append(tf)
    temp_genes+=collectri.loc[collectri['source']==tf]['target'].tolist()
temp_genes = list(set(temp_genes))
print(temp_genes)

temp_genes = []
for tf in list(set([a for a in gene_sample_hits.keys()])):
    temp_genes.append(tf)
    temp_genes+=collectri.loc[collectri['source']==tf]['target'].tolist()
temp_genes = list(set(temp_genes))
print(temp_genes)

temp_genes = []
for tf in list(set([a for a in list(set(gene_multi_sample_hits.keys()).difference(set(gene_hits.keys()).union(set(gene_sample_hits.keys()))))])):
    temp_genes.append(tf)
    temp_genes+=collectri.loc[collectri['source']==tf]['target'].tolist()
temp_genes = list(set(temp_genes))
print(temp_genes)

In [ ]:
all_DE_regulon_keys = {}
for cnp_type in all_DE_regulons.keys():
    for sample in all_DE_regulons[cnp_type].keys():
        if sample not in all_DE_regulon_keys.keys():
            all_DE_regulon_keys[sample] = []
        all_DE_regulon_keys[sample] = list(set(all_DE_regulon_keys[sample]+list(all_DE_regulons[cnp_type][sample].keys())))

print(sum([len(b.keys()) for a in all_DE_regulons.values() for b in a.values()]))

all_non_DE_regulon_keys = {}
for cnp_type in all_non_DE_regulons.keys():
    for sample in all_non_DE_regulons[cnp_type].keys():
        if sample not in all_non_DE_regulon_keys.keys():
            all_non_DE_regulon_keys[sample] = []
        all_non_DE_regulon_keys[sample] = list(set(all_non_DE_regulon_keys[sample]+list(all_non_DE_regulons[cnp_type][sample].keys())))

print(sum([len(b.keys()) for a in all_non_DE_regulons.values() for b in a.values()]))

DE_regulon_keys = {}
for cnp_type in DE_regulons.keys():
    for sample in DE_regulons[cnp_type].keys():
        if sample not in DE_regulon_keys.keys():
            DE_regulon_keys[sample] = []
        DE_regulon_keys[sample] = list(set(DE_regulon_keys[sample]+list(DE_regulons[cnp_type][sample].keys())))

print(sum([len(b.keys()) for a in DE_regulons.values() for b in a.values()]))

In [ ]:
new_order = [x for x in regulon_score.obs['sample_clone'].unique() if x[-1] != 'N'] + [x for x in regulon_score.obs['sample_clone'].unique() if x[-1] == 'N']
regulon_score.obs['sample_clone'] = regulon_score.obs['sample_clone'].cat.reorder_categories(new_order)
temp = [x for x in list(regulon_score_clone.obs['sample_clone'].unique()) if x[-1]!='N']
#temp.sort()

#clone level selected tfs
#clone_tfs = set(['PRDM2', 'ZKSCAN7', 'SMARCC1', 'BRD4', 'ETV3', 'FOXJ1', 'CREBZF', 'HHEX', 'NSD2', 'TEF', 'FOXQ1', 'PAX2', 'NFE2L3', 'CTBP2', 'NSD2', 'JARID2', 'ISL1', 'NFE2', 'PEG3', 'FGF2', 'SOX10', 'NFE2L3', 'ETV1', 'ID4', 'TGIF1'])
#print(temp)

In [ ]:
DE_regulon_selected_keys = {'CID45131': ['PRDM2'], 'CID45150': ['ZKSCAN7'], 'CID449711': ['SMARCC1'], 'CID449710': ['BRD4'], 'CID449712': ['FOXJ1', 'ISL1', 'ETV3'], 'CID449911': ['CREBZF'], 'Triple_negative_01260': ['PAX2', 'TEF'], 'Triple_negative_01261': ['HHEX', 'FOXQ1', 'NFE2L3'], 'Triple_negative_01262': ['CTBP2', 'NSD2'], 'Triple_negative_BRCA1_01311': ['JARID2', 'ISL1'], 'Triple_negative_BRCA1_05540': ['NFE2', 'CTBP2'], 'TNBC11': ['NFE2', 'FGF2', 'PEG3'], 'TNBC10': ['ETV1', 'HHEX', 'SOX10', 'NFE2L3'], 'TNBC21': ['ETV1', 'CTBP2', 'NSD2'], 'TNBC20': ['ID4', 'ZKSCAN7', 'TGIF1']}
all_DE_regulon_selected_keys = {'CID44991T': ['GRHL2', 'CREBZF'], 'CID4513T': ['KLF11'], 'CID4523T': ['ZNF335'], 'TNBC1T': ['PEG3', 'NFE2', 'ETV1', 'SOX10'], 'TNBC2T': ['ID4', 'ZKSCAN7', 'CTBP2', 'NSD2'], 'Triple_negative_0126T': ['HHEX', 'NFE2L3', 'TEF', 'CTBP2'], 'TNBC3T': ['MED1', 'HOXB1', 'CTBP2', 'NSD2']}
all_non_DE_regulon_selected_keys = {'CID4495T': ['ETV3', 'TFAM', 'SMARCC1', 'E2F6'], 'CID44971T': ['MAFK', 'JARID2', 'HTATIP2', 'GRHL2'], 'CID44991T': ['NSD2', 'SMARCC1', 'LYL1'], 'CID4513T': ['TBX3', 'ZFP42'], 'CID4515T': ['TRPS1', 'SP4', 'ISL1', 'MECP2'], 'CID4523T': ['TADA2A', 'TFAM', 'SIM2'], 'TNBC1T': ['GATA4', 'TCF7', 'PAX6'], 'TNBC2T': ['ZBTB33', 'RXRB', 'ID4'], 'Triple_negative_0126T': ['NONO', 'POU2F3', 'TEAD4'], 'Triple_negative_BRCA1_0131T': ['FOXG1', 'ZBTB33'], 'Triple_negative_BRCA1_0554T': ['BHLHE41', 'NR5A2', 'PKNOX1', 'NSD2'], 'TNBC3T': ['NR2C1', 'SALL4']}

In [ ]:
wang_regulons = {'DE_regulon_keys':DE_regulon_keys.copy(), 'all_DE_regulon_keys':all_DE_regulon_keys.copy(), 'all_non_DE_regulon_keys':all_non_DE_regulon_keys.copy()}

In [ ]:
glio_regulons = {'DE_regulon_keys':DE_regulon_keys.copy(), 'all_DE_regulon_keys':all_DE_regulon_keys.copy(), 'all_non_DE_regulon_keys':all_non_DE_regulon_keys.copy()}

In [ ]:
DE_regulon_keys = wang_regulons['DE_regulon_keys'] | glio_regulons['DE_regulon_keys']
all_DE_regulon_keys = wang_regulons['all_DE_regulon_keys'] | glio_regulons['all_DE_regulon_keys']
all_non_DE_regulon_keys = wang_regulons['all_non_DE_regulon_keys'] | glio_regulons['all_non_DE_regulon_keys']

In [ ]:
DE_regulon_keys

In [ ]:
#sc.pl.rank_genes_groups_matrixplot(adata=regulon_score, groupby='sample', n_genes=10, min_logfoldchange=1, standard_scale='var')
#sc.pl.rank_genes_groups_heatmap(adata=regulon_score, groupby='sample', n_genes=20, min_logfoldchange=2, standard_scale='var', show_gene_labels=True)
#sc.pl.heatmap(adata=regulon_score, var_names=temp_df[['group', 'name']].groupby('group')['name'].apply(list).to_dict(), groupby='sample', standard_scale='var', show_gene_labels=True)
#sc.pl.heatmap(adata=regulon_score, var_names=temp_df[['group', 'name']].groupby('group')['name'].apply(list).to_dict(), groupby='sample', standard_scale='var', show_gene_labels=True)

#sc.pl.matrixplot(adata=regulon_score, var_names={x:list(set(DE_regulon_keys[x])) for x in temp if x in DE_regulon_keys.keys()}, groupby='sample_clone', figsize=(16,12), standard_scale='var')
sc.pl.matrixplot(adata=regulon_score, var_names={k:all_DE_regulon_keys[k] for k in regulon_score.obs['sample_tumor'].cat.categories if k in all_DE_regulon_keys.keys()}, groupby='sample_tumor', standard_scale='var')
sc.pl.matrixplot(adata=regulon_score, var_names={k:all_non_DE_regulon_keys[k] for k in regulon_score.obs['sample_tumor'].cat.categories if k in all_non_DE_regulon_keys.keys()}, groupby='sample_tumor', standard_scale='var')

#mask_keep = ~regulon_score.obs['sample_clone'].isin([x for x in regulon_score.obs['sample_clone'].unique() if x[-1]=='N'])
#TNBC
#sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names={k:DE_regulon_selected_keys[k] for k in regulon_score.obs['sample_clone'].cat.categories if k in DE_regulon_selected_keys.keys()}, groupby='sample_clone', figsize=(16,12), standard_scale='var')
#sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names={k:all_DE_regulon_selected_keys[k] for k in regulon_score.obs['sample_tumor'].cat.categories if k in all_DE_regulon_selected_keys.keys()}, groupby='sample_tumor', standard_scale='var')
#sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names={k:all_non_DE_regulon_selected_keys[k] for k in regulon_score.obs['sample_tumor'].cat.categories if k in all_non_DE_regulon_selected_keys.keys()}, groupby='sample_tumor', standard_scale='var')

#Glio
#sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names={k:DE_regulon_selected_keys[k] for k in regulon_score.obs['sample_clone'].cat.categories if k in DE_regulon_selected_keys.keys()}, groupby='sample_clone', figsize=(16,12), standard_scale='var')
#sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names={k:all_DE_regulon_selected_keys[k] for k in regulon_score.obs['sample_tumor'].cat.categories if k in all_DE_regulon_selected_keys.keys()}, groupby='sample_tumor', standard_scale='var')
#sc.pl.matrixplot(adata=regulon_score[mask_keep], var_names={k:all_non_DE_regulon_selected_keys[k] for k in regulon_score.obs['sample_tumor'].cat.categories if k in all_non_DE_regulon_selected_keys.keys()}, groupby='sample_tumor', standard_scale='var')

In [ ]:
'''
clone level pathway enrichment:  https://biit.cs.ut.ee/gplink/l/aM797LXu5Qd
sample level pathway enrichment: https://biit.cs.ut.ee/gplink/l/a-5TBpQUaR8
sample level non DE pathway enrichment: https://biit.cs.ut.ee/gplink/l/a1oY02eHFTh
'''

temp_genes = []
for tf in list(set([a for x in temp if x in DE_regulon_selected_keys.keys() for a in list(set(DE_regulon_selected_keys[x]).intersection(clone_tfs))])):
    temp_genes.append(tf)
    temp_genes+=collectri.loc[collectri['source']==tf]['target'].tolist()
temp_genes = list(set(temp_genes))
print(temp_genes)
print(list(set([a for x in temp if x in DE_regulon_selected_keys.keys() for a in DE_regulon_selected_keys[x]])))

temp_genes = []
for tf in [a for x in all_DE_regulon_selected_keys.keys() for a in all_DE_regulon_selected_keys[x]]:
    temp_genes.append(tf)
    temp_genes+=collectri.loc[collectri['source']==tf]['target'].tolist()
temp_genes = list(set(temp_genes))
print(temp_genes)
print(list(set([a for x in all_DE_regulon_selected_keys.keys() for a in all_DE_regulon_selected_keys[x]])))

temp_genes = []
for tf in [a for x in all_non_DE_regulon_selected_keys.keys() for a in all_non_DE_regulon_selected_keys[x]]:
    temp_genes.append(tf)
    temp_genes+=collectri.loc[collectri['source']==tf]['target'].tolist()
temp_genes = list(set(temp_genes))
print(temp_genes)
print(list(set([a for x in all_non_DE_regulon_selected_keys.keys() for a in all_non_DE_regulon_selected_keys[x]])))

In [ ]:
{x:list(set(DE_regulon_keys[x]).intersection(clone_tfs)) for x in temp if x in DE_regulon_keys.keys()}

In [ ]:
[a for x in temp if x in all_DE_regulon_keys.keys() for a in all_DE_regulon_keys[x]]

In [ ]:
temp_df = []
for i in range(len(regulon_score.uns['rank_genes_groups']['logfoldchanges'])):
    for j in range(len(regulon_score.uns['rank_genes_groups']['logfoldchanges'][i])):
        if (regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j] > 3) and regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.005:
            temp_df.append({'group':regulon_score.uns['rank_genes_groups']['names'].dtype.names[j], 'name':regulon_score.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':regulon_score.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':regulon_score.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':regulon_score.uns['rank_genes_groups']['pvals_adj'][i][j]})

temp_df = pd.DataFrame(temp_df).sort_values(by=['group', 'logfoldchange'], ascending=[True, False])#.to_csv(DATA_ROOT + '/temp_DE_sample_regulons.csv', index=False)

In [ ]:
print(temp_df.to_string())
#np.histogram(temp_df['logfoldchange'])

In [ ]:
import copy

'''
def merge_dicts_with_DE_suffix(dict1, dict2):
    # Deep copy so we don't modify original dict2
    merged = copy.deepcopy(dict2)

    for sample, categories in dict1.items():

        # Ensure sample exists in merged
        if sample not in merged:
            merged[sample] = {}

        for cat, tf_dict in categories.items():
            for tf, value in tf_dict.items():

                new_key = f"{tf}_DE"

                # If TF exists in dict2 → rename existing key
                if tf in merged[sample]:
                    merged[sample][new_key] = merged[sample].pop(tf)
                else:
                    # Add new TF from dict1
                    merged[sample][new_key] = value

    return merged
'''

def merge_tf_targets(dict1, dict2):
    dict3 = {}

    # All samples from both dictionaries
    all_samples = set(dict1['cnp'].keys()) | set(dict1['non_cnp'].keys()) | set(dict2['cnp'].keys()) | set(dict2['non_cnp'].keys())

    for cnp_type in ['cnp', 'non_cnp']:
        for sample in all_samples:
            if sample not in dict3.keys():
                dict3[sample] = {}

            # ---- Add from dict2 (already flat) ----
            if sample in dict2[cnp_type]:
                for tf, tf_data in dict2[cnp_type][sample].items():
                    dict3[sample][tf] = tf_data["targets"]

            # ---- Add from dict1 (nested under categories) ----
            if sample in dict1[cnp_type]:
                for cat in dict1[cnp_type][sample].values():
                    for tf, tf_data in cat.items():
                        dict3[sample][tf] = tf_data["targets"]

    return dict3

def select_first_n_tfs(merged_dict, n):
    selected = {}

    for sample, tf_dict in merged_dict.items():
        selected[sample] = {}

        # Separate keys
        non_de = [k for k in tf_dict.keys() if not k.endswith("_DE")]
        de = [k for k in tf_dict.keys() if k.endswith("_DE")]

        # Take first n of each
        chosen_keys = non_de[:n] + de[:n]

        # Build new dictionary
        for key in chosen_keys:
            selected[sample][key] = tf_dict[key]

    return selected

sample_clone_DE_regulons = merge_tf_targets(DE_regulons, all_DE_regulons)
#selected_regulons = select_first_n_tfs(sample_clone_DE_regulons, 8)


In [ ]:
len(sample_clone_DE_regulons['MGH152'])

In [ ]:
#sc.pl.matrixplot(regulon_score, {k:[x.replace('_DE', '') for x in v.keys()] for k,v in selected_regulons.items()}, groupby='sample_clone', standard_scale='var')
regulon_keys = {k:[x for x in v.keys()] for k,v in sample_clone_DE_regulons.items()}
mp = sc.pl.matrixplot(regulon_score, regulon_keys, groupby='sample_clone', standard_scale='var', swap_axes=True, return_fig=True)
mp_df = mp.values_df.T.copy()

cat_colormap = {#'sample':{'MGH102':'#006400', 'MGH105':'#ff0000', 'MGH122':'#ffd700', 'MGH125':'#c71585', 'MGH136':'#0F00FF', 'MGH152':'#1e90ff', 'SF10022': '#00CED1', 'SF11979sn':'#FF8C00', 'SF4297':'#8A2BE2', 'SF6996':'#2E8B57', 'SF9259R':'#DC143C'}, 
                'sample':{'CID4495':'#006400', 'CID4513':'#ff0000', 'CID4515':'#ffd700', 'CID4523':'#c71585', 'CID44971':'#0F00FF', 'CID44991':'#1e90ff', 'Triple_negative_0126':'#00CED1', 'Triple_negative_BRCA1_0131':'#FF8C00', 'Triple_negative_BRCA1_0554':'#8A2BE2', 'TNBC1':'#2E8B57', 'TNBC2':'#DC143C', 'TNBC3':'#ffa500'},
                'clone':{'color':'#b03060','cnp':'#ffa500', 'non_cnp':'#008080'}, 
                'non_clone':{'color':'#ffdab9','cnp':'#ffa500', 'non_cnp':'#008080'}, 
                'both':{
                    'color':'#6495ed',
                    'cnp_clone':{'cnp_non_clone':'#9c1663', 'non_cnp_non_clone':'#7fa200'}, 
                    'non_cnp_clone':{'cnp_non_clone':'#ff3b83', 'non_cnp_non_clone':'#f65900'}
                    }
                }

regulon_color = {'sample':[], 'clone':[], 'cnp':[]}
for k,v in regulon_keys.items():
    if k in all_DE_regulons['cnp'].keys() or k in all_DE_regulons['non_cnp'].keys():
        for tf in v:
            regulon_color['sample'].append(cat_colormap['sample'][k])
            presence = {'clone':{'cnp':False, 'non_cnp':False}, 'sample':{'cnp':False, 'non_cnp':False}}
            for cnp_type in ['cnp', 'non_cnp']:
                if k in DE_regulons[cnp_type].keys():
                    for c in DE_regulons[cnp_type][k].keys():
                        if tf in DE_regulons[cnp_type][k][c].keys():
                            presence['clone'][cnp_type] = True
                if k in all_DE_regulons[cnp_type].keys():
                    if tf in all_DE_regulons[cnp_type][k].keys():
                        presence['sample'][cnp_type] = True
            total_true = sum(val for subdict in presence.values() for val in subdict.values())
            clone_true = sum(val for val in presence['clone'].values())
            sample_true = sum(val for val in presence['sample'].values())
            if total_true == 2 and clone_true == 1 and sample_true == 1:
                regulon_color['clone'].append(cat_colormap['both']['color'])
                if presence['clone']['cnp'] and presence['sample']['cnp']:
                    regulon_color['cnp'].append(cat_colormap['both']['cnp_clone']['cnp_non_clone'])
                elif presence['clone']['cnp'] and presence['sample']['non_cnp']:
                    regulon_color['cnp'].append(cat_colormap['both']['cnp_clone']['non_cnp_non_clone'])
                elif presence['clone']['non_cnp'] and presence['sample']['cnp']:
                    regulon_color['cnp'].append(cat_colormap['both']['non_cnp_clone']['cnp_non_clone'])
                elif presence['clone']['non_cnp'] and presence['sample']['non_cnp']:
                    regulon_color['cnp'].append(cat_colormap['both']['non_cnp_clone']['non_cnp_non_clone'])
                else:
                    print('NON EXISTENT SCENARIO:',tf,k,presence)
            elif total_true < 2 and sample_true == 1:
                regulon_color['clone'].append(cat_colormap['non_clone']['color'])
                if presence['sample']['cnp']:
                    regulon_color['cnp'].append(cat_colormap['non_clone']['cnp'])
                elif presence['sample']['non_cnp']:
                    regulon_color['cnp'].append(cat_colormap['non_clone']['non_cnp'])
                else:
                    print('NON EXISTENT SCENARIO:',tf,k,presence)
            elif total_true < 2 and clone_true == 1:
                regulon_color['clone'].append(cat_colormap['clone']['color'])
                if presence['clone']['cnp']:
                    regulon_color['cnp'].append(cat_colormap['clone']['cnp'])
                elif presence['clone']['non_cnp']:
                    regulon_color['cnp'].append(cat_colormap['clone']['non_cnp'])
                else:
                    print('NON EXISTENT SCENARIO:',tf,k,presence)
            elif total_true == 2 and clone_true == 2:
                regulon_color['clone'].append(cat_colormap['clone']['color'])
                if presence['clone']['cnp']:
                    regulon_color['cnp'].append(cat_colormap['clone']['cnp'])
                else:
                    print('NON EXISTENT SCENARIO:',tf,k,presence)
            else:
                print('NON EXISTENT SCENARIO:',tf,k,presence)
            
regulon_color = pd.DataFrame(regulon_color)
regulon_color.index = mp_df.index
new_col_order = []
for k in regulon_keys.keys():
    for col in mp_df.columns:
        if col[0:-1] == k:
            new_col_order.append(col)

sample_color = [cat_colormap['sample'][x[0:-1]] for x in new_col_order]

g = sns.clustermap(mp_df[new_col_order], row_colors=regulon_color, col_colors=sample_color, row_cluster=False, col_cluster=False, cmap='viridis', xticklabels=True, yticklabels=True, figsize=(12,180))  
g.figure.savefig(DATA_ROOT + "/manuscript_figs/regulons_clustermap_TNBC.pdf", format="pdf", dpi=300)
plt.close(g.figure)

In [ ]:
sum([len(x) for x in regulon_keys.values()])

## Pathway analysis

In [ ]:
geneset = list(set(collectri.loc[collectri['source']=='SMAD6']['target'].tolist()+['SMAD6']))
gp = GProfiler(return_dataframe=True)
enrichment = gp.profile(organism='hsapiens', query=geneset, no_evidences=False)

In [ ]:
geneset

In [ ]:
enrichment.loc[enrichment['source'].isin(['WP','KEGG'])]

In [ ]:
print(dc.op.show_resources()['name'].tolist())

In [ ]:
#Pathway enrichment across clones
#pathway = dc.op.hallmark(organism="human")
pathway = pd.read_csv(DATA_ROOT + '/msig_human_WP_H_KEGG_new.csv')
subset_ids = [x.split(':')[1] for x in enrichment['native'].unique()]
#pathway = pathway.loc[]
pathway = pathway[['gs_name', 'human_gene_symbol']]
pathway.columns = ['source', 'target']
#Add meta modules
pathway = pathway.to_dict(orient='list')
modules = pd.read_csv(DATA_ROOT + '/Glio/Glio_meta_modules.csv').to_dict(orient='list')
module_related_genes = {}
for module, genes in modules.items():
    module_related_genes[module] = {'source':[],'target':[]}
    subnet = collectri.loc[(collectri['source'].isin(genes)) | (collectri['target'].isin(genes))]
    module_related_genes[module]['source'] = list(set(subnet['source'].tolist()))
    module_related_genes[module]['target'] = list(set(subnet['target'].tolist()))
    related_genes = list(set(subnet['source'].tolist()+subnet['target'].tolist()))
    for gene in related_genes:
        pathway['source'].append(module)
        pathway['target'].append(gene)
pathway = pd.DataFrame(pathway)
pathway = pathway.drop_duplicates()

dc.mt.ulm(data=adata, net=pathway)
pathway_scores = dc.pp.get_obsm(adata=adata, key="score_ulm")
pathway_scores_df = dc.tl.rankby_group(adata=pathway_scores, groupby="inferred_clone")
pathway_scores_df = pathway_scores_df[(pathway_scores_df["stat"] > 0) & (pathway_scores_df['padj'] < 0.05)]#may have to use padj?
#pathway_scores_df = pathway_scores_df.loc[pathway_scores_df['name'].isin(['MES2', 'MES1', 'AC', 'OPC', 'NPC1', 'NPC2', 'G1/S', 'G2/M'])]

In [ ]:
pathway = pd.read_csv(DATA_ROOT + '/msig_human_WP_H_KEGG_new.csv')
subset_ids = []
for i,row in enrichment.iterrows():
    if row['source'] == 'KEGG':
        subset_ids.append('hsa'+row['native'].split(':')[1])
    elif row['source'] == 'WP':
        subset_ids.append(row['native'].split(':')[1])

In [ ]:
pathway.loc[pathway['gs_exact_source'].isin(subset_ids)]['gs_name'].unique()

In [ ]:
pathway_scores_df

In [ ]:
n_markers = 200
pathway_markers = (
    pathway_scores_df.groupby("group")
    .head(n_markers)
    .drop_duplicates("name")
    .groupby("group")["name"]
    .apply(lambda x: list(x))
    .to_dict()
)
pathway_markers

In [ ]:
sc.pl.dotplot(
    adata=pathway_scores,
    var_names=pathway_markers,#['MES2', 'MES1', 'AC', 'OPC', 'NPC1', 'NPC2', 'G1/S', 'G2/M'],#
    groupby="inferred_clone",
    vcenter=0.5,
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
)

In [ ]:
#Use only to backup original regulon_score_df !!
#regulon_score_df_all = regulon_score_df.copy()

selected_rows = []
for i, row in regulon_score_df.iterrows():
    if row['name'] in regulon_markers[row['group']]:
        selected_rows.append(i)

regulon_score_df = regulon_score_df.loc[selected_rows,:].copy()

In [ ]:
def unilateral_jaccard(pathway_name, regulon_name, pathway_df, collectri_df):
    A = set(pathway_df.loc[pathway_df['source'] == pathway_name, 'target'])
    B = set(collectri_df.loc[collectri_df['source'] == regulon_name, 'target'])

    if len(B) == 0:
        return 0.0

    return len(A & B) / len(B)


jaccard_records = []
padj_records = []

for group in sorted(set(pathway_scores_df['group']).intersection(regulon_score_df['group'])):

    pw_sub = pathway_scores_df[pathway_scores_df['group'] == group]
    reg_sub = regulon_score_df[regulon_score_df['group'] == group]

    for _, pw_row in pw_sub.iterrows():
        for _, reg_row in reg_sub.iterrows():

            jac = unilateral_jaccard(
                pw_row['name'],
                reg_row['name'],
                pathway,
                collectri
            )

            jaccard_records.append({
                "pathway": pw_row['name'],
                "regulon": reg_row['name'],
                "group": group,
                "value": jac
            })

            padj_records.append({
                "pathway": pw_row['name'],
                "regulon": reg_row['name'],
                "group": group,
                "value": -1*np.log10(pw_row['padj'])
            })


jaccard_df = pd.DataFrame(jaccard_records)
padj_df = pd.DataFrame(padj_records)

# Pivot to matrices
jaccard_mat = jaccard_df.pivot(
    index="pathway", columns="regulon", values="value"
).fillna(0)

padj_mat = padj_df.pivot(
    index="pathway", columns="regulon", values="value"
).fillna(1.0)


pr_rel = sc.AnnData(
    X=jaccard_mat.values,
    obs=pd.DataFrame(index=jaccard_mat.index),
    var=pd.DataFrame(index=jaccard_mat.columns)
)

# Store padj as dot size matrix
pr_rel.layers["padj"] = padj_mat.values
pr_rel.obs['dummy'] = pr_rel.obs.index

In [ ]:
def get_second_half(name):
    return str(name).split('_')[-1] # Adjust the slicing index [4:] based on your exact naming convention


pathway_ordered = (
    pathway_scores_df
    .sort_values(["group", "name"])
)

pathway_ordered = pathway_ordered.loc[pathway_ordered['name'].isin(pr_rel.obs_names)]

row_names = pathway_ordered["name"].tolist()
row_groups = pathway_ordered["group"].tolist()

regulon_ordered = (
    regulon_score_df
    .sort_values(["group", "name"])
)

regulon_ordered = regulon_ordered.loc[regulon_ordered['name'].isin(pr_rel.var_names)]

col_names = regulon_ordered["name"].tolist()
col_groups = regulon_ordered["group"].tolist()

pr_rel = pr_rel.to_df().loc[row_names, col_names].copy()
pr_rel.index = [row_names[x]+'_'+row_groups[x] for x in range(len(row_names))]
pr_rel.columns = [col_names[x]+'_'+col_groups[x] for x in range(len(col_names))]
pr_rel = sc.AnnData(pr_rel)
pr_rel.obs["pathway"] = pr_rel.obs_names
pr_rel.obs['pathway'] = pd.Categorical(pr_rel.obs["pathway"], categories=pr_rel.obs_names, ordered=True)
pr_rel.obs["group"] = row_groups
pr_rel.var["regulon"] = pr_rel.var_names
pr_rel.var["regulon"] = pd.Categorical(pr_rel.var["regulon"], categories=pr_rel.var_names, ordered=True)
pr_rel.var["group"] = col_groups

jaccard_mat = jaccard_mat.loc[row_names, col_names]
jaccard_mat.index = [row_names[x]+'_'+row_groups[x] for x in range(len(row_names))]
jaccard_mat.columns = [col_names[x]+'_'+col_groups[x] for x in range(len(col_names))]
padj_mat = padj_mat.loc[row_names, col_names]
padj_mat.index = [row_names[x]+'_'+row_groups[x] for x in range(len(row_names))]
padj_mat.columns = [col_names[x]+'_'+col_groups[x] for x in range(len(col_names))]

# Create a boolean mask where the condition is True if they do NOT match
mask = np.zeros(jaccard_mat.shape, dtype=bool)

for i, row_name in enumerate(jaccard_mat.index):
    for j, col_name in enumerate(jaccard_mat.columns):
        if get_second_half(row_name) != get_second_half(col_name):
            mask[i, j] = True

# Use .mask() to replace values where the mask is True with 0
jaccard_mat = jaccard_mat.mask(mask, 0)
padj_mat = padj_mat.mask(mask, 0)

def group_positions(groups):
    positions = []
    labels = []
    start = 0
    for g in sorted(set(groups)):
        count = groups.count(g)
        positions.append((start, start + count - 1))
        labels.append(f"Group {g}")
        start += count
    return positions, labels

var_group_positions, var_group_labels = group_positions(col_groups)

In [ ]:
dp = sc.pl.dotplot(
    pr_rel,
    var_names=pr_rel.var_names,
    groupby="pathway",
    dot_color_df=jaccard_mat,
    dot_size_df=padj_mat,
    var_group_positions=var_group_positions,
    var_group_labels=var_group_labels,
    color_map="RdBu_r",
    figsize=(50, 40),
    show=False,
    size_title=None,
    vmax=0.5,
    vcenter=0.1,
    dendrogram=False
)

ax = dp['mainplot_ax']
ax.set_title("Pathway x Regulon Jaccard", fontsize=16)
ax.tick_params(axis='both', labelsize=12)

row_group_change = np.where(np.diff([int(x) for x in row_groups]))[0]
for idx in row_group_change:
    ax.axhline(idx + 1, color="black", linewidth=1)


size_ax = dp.get('size_legend_ax')
if size_ax is not None:
    size_ax.clear()

    bins = [0, 5, 10, 20, np.inf]
    labels = ["0-5", "5-10", "10-20", ">20"]
    size_vals = [3, 8, 15, 25]

    for s, lab in zip(size_vals, labels):
        size_ax.scatter([], [], s=s*10, color='gray', label=lab)

    size_ax.legend(
        title="-log10(adj p)",
        frameon=False,
        loc="center",
        fontsize=10,
        title_fontsize=12
    )

plt.tight_layout()
plt.show()

In [ ]:
np.histogram(pr_rel.to_df())

In [ ]:
#Pathway enrichment all samples
#all_data = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
#           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}
all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 
            'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}

DE_pathways = {'cnp':{}, 'non_cnp':{}}
sample_DECNP_genes = {}
#cnp_thresh_all = {'MGH105':3, 'MGH102':2, 'MGH122':2, 'MGH125':2, 'MGH136':2, 'MGH152':3, 'SF10022':3, 'SF11979sn':3, 'SF4297':3, 'SF6996':3, 'SF9259R':3}
cnp_thresh_all = {'CID4495':3, 'CID4513':1, 'CID4515':2, 'CID4523':1, 'CID44971':3, 'CID44991':3, 'Triple_negative_0126':3, 'Triple_negative_BRCA1_0131':3, 'Triple_negative_BRCA1_0554':3, 'TNBC1':3, 'TNBC2':3, 'TNBC3':2}

for specimen, SAMPLES in all_data.items():
    for SAMPLE in SAMPLES:
        DE_pathways['cnp'][SAMPLE] = {}
        DE_pathways['non_cnp'][SAMPLE] = {}
        sample_DECNP_genes[SAMPLE] = {}
        #specimen = 'Glio/Data_Neftel2019_Brain/SmartSeq2'
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone.h5ad'
        else:
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'

        gene_order = pd.read_csv(GENE_ORDER_FILE, names = range(5), sep='\t')
        gene_order = gene_order.sort_values(2)[0].tolist()

        adata = sc.read_h5ad(SAMPLE_PATH)
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            adata_dna = sc.AnnData(adata.uns['CNA'])
        else:
            adata_dna = sc.AnnData(adata.uns['cnp'])
        adata_dna.obs_names = adata.obs_names.copy()
        adata_dna.var_names = adata.var_names.copy()
        adata_dna.obs = adata.obs.copy()
        adata_dna.var = adata.var.copy()

        clones = adata.obs.inferred_clone.unique()
        for clone in clones:
            if clone=='N' or (specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2' and clone=='0') or (specimen=='Wang_2019' and clone=='2') or len(adata.obs.loc[adata.obs['inferred_clone']==clone]) < 20:
                adata = adata[adata.obs.loc[adata.obs['inferred_clone']!=clone].index].copy()
                
        print(adata.obs['inferred_clone'].unique())
        
        sc.pp.filter_cells(adata, min_genes=100)
        sc.pp.filter_genes(adata, min_cells=3)
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        sc.tl.pca(adata)
        sc.pp.neighbors(adata)
        sc.tl.umap(adata)
        
        adata_dna = adata_dna[adata.obs_names, adata.var_names]
        adata_dna = adata_dna[adata_dna.obs.loc[adata_dna.obs['inferred_clone']!='N'].index].copy()
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            adata_dna = adata_dna[adata_dna.obs.loc[adata_dna.obs['inferred_clone']!='0'].index].copy()
        elif specimen=='Wang_2019':
            adata_dna = adata_dna[adata_dna.obs.loc[adata_dna.obs['inferred_clone']!='2'].index].copy()
        dna_copy = adata_dna.copy()
        sc.tl.rank_genes_groups(dna_copy, groupby="inferred_clone", method="wilcoxon")
        
        cnp_thresh = cnp_thresh_all[SAMPLE]

        DECNP_genes = {x:[] for x in adata.obs['inferred_clone'].unique()}
        for i in range(len(dna_copy.uns['rank_genes_groups']['logfoldchanges'])):
            for j in range(len(dna_copy.uns['rank_genes_groups']['logfoldchanges'][i])):
                if (dna_copy.uns['rank_genes_groups']['logfoldchanges'][i][j] > cnp_thresh or dna_copy.uns['rank_genes_groups']['logfoldchanges'][i][j] < -1*cnp_thresh) and dna_copy.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
                    DECNP_genes[dna_copy.uns['rank_genes_groups']['names'].dtype.names[j]].append(dna_copy.uns['rank_genes_groups']['names'][i][j])
        
        sample_DECNP_genes[SAMPLE] = [item for sublist in DECNP_genes.values() for item in sublist]
        
        
        #Pathway enrichment across clones
        #pathway = dc.op.hallmark(organism="human")
        #Add meta modules
        pathway = pd.read_csv(DATA_ROOT + '/WP_KEGG_pathways.csv')
        pathway = pathway[['pathway_id','gene_symbol']]
        pathway.columns = ['source', 'target']
        pathway = pathway.drop_duplicates()

        dc.mt.ulm(data=adata, net=pathway)
        pathway_scores = dc.pp.get_obsm(adata=adata, key="score_ulm")
        sc.tl.rank_genes_groups(adata=pathway_scores, groupby="inferred_clone")
        
        pathway_score_df = []
        for i in range(len(pathway_scores.uns['rank_genes_groups']['logfoldchanges'])):
            for j in range(len(pathway_scores.uns['rank_genes_groups']['logfoldchanges'][i])):
                if (pathway_scores.uns['rank_genes_groups']['logfoldchanges'][i][j] > 0.25) and pathway_scores.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
                    pathway_score_df.append({'group':pathway_scores.uns['rank_genes_groups']['names'].dtype.names[j], 'name':pathway_scores.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':pathway_scores.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':pathway_scores.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':pathway_scores.uns['rank_genes_groups']['pvals_adj'][i][j]})

        if len(pathway_score_df) > 0:
            pathway_score_df = pd.DataFrame(pathway_score_df).sort_values(by='group')
            pathway_markers = {x:[] for x in adata.obs['inferred_clone'].unique()}
            threshold = 2000
            cnp_gene_thresh = 1
            select_top_n = 20
            
            de_count = {x:0 for x in adata.obs['inferred_clone'].unique()}
            decnp_count = {x:0 for x in adata.obs['inferred_clone'].unique()}

            for i,row in pathway_score_df.iterrows():
                targets = pathway.loc[pathway['source']==row['name']]['target'].tolist()
                selected_targets = []
                for target in targets:
                    if target in DECNP_genes[row['group']]:
                        if DECNP_genes[row['group']].index(target) <= threshold:
                            selected_targets.append(target)
                if len(selected_targets) >= cnp_gene_thresh:
                    if decnp_count[row['group']] < select_top_n:
                        if row['group'] not in DE_pathways['cnp'][SAMPLE].keys():
                            DE_pathways['cnp'][SAMPLE][row['group']] = {}
                        DE_pathways['cnp'][SAMPLE][row['group']][row['name']] = {'targets':targets, 'cnp_targets':selected_targets}
                        pathway_markers[row['group']].append(row['name'])
                        decnp_count[row['group']]+=1
                elif de_count[row['group']]  < select_top_n:
                    if row['group'] not in DE_pathways['non_cnp'][SAMPLE].keys():
                        DE_pathways['non_cnp'][SAMPLE][row['group']] = {}
                    DE_pathways['non_cnp'][SAMPLE][row['group']][row['name']] = {'targets':targets}
                    de_count[row['group']]+=1


In [ ]:
#datasets = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
#           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}
datasets = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 
            'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}

all_DE_pathways = {'cnp':{}, 'non_cnp':{}}
all_adata = {}

for specimen, SAMPLES in datasets.items():
    for SAMPLE in SAMPLES:
        #specimen = 'Glio/Data_Neftel2019_Brain/SmartSeq2'
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone.h5ad'
        else:
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'
        adata = sc.read_h5ad(SAMPLE_PATH)
        clones = adata.obs.inferred_clone.unique()
        for clone in clones:
            if clone=='N' or (specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2' and clone=='0') or (specimen=='Wang_2019' and clone=='2') or len(adata.obs.loc[adata.obs['inferred_clone']==clone]) < 20:
                adata = adata[adata.obs.loc[adata.obs['inferred_clone']!=clone].index].copy()
        adata.obs['sample'] = SAMPLE
        adata.obs['sample_clone'] = SAMPLE+adata.obs['inferred_clone'].astype(str)
        all_adata[SAMPLE] = adata.copy()

adata_combined = sc.concat([x for x in all_adata.values()], join="outer")
adata_combined.obs['sample'] = adata_combined.obs['sample'].astype('category')
adata_combined.obs['sample_clone'] = adata_combined.obs['sample_clone'].astype('category')
sc.pp.filter_cells(adata_combined, min_genes=100)
sc.pp.filter_genes(adata_combined, min_cells=3)
sc.pp.normalize_total(adata_combined)
sc.pp.log1p(adata_combined)
sc.pp.highly_variable_genes(adata_combined, n_top_genes=2000)
sc.tl.pca(adata_combined)
sc.pp.neighbors(adata_combined)
sc.tl.umap(adata_combined)

#Pathway enrichment across clones
#pathway = dc.op.hallmark(organism="human")
pathway = pd.read_csv(DATA_ROOT + '/WP_KEGG_pathways.csv')
pathway = pathway[['pathway_id','gene_symbol']]
pathway.columns = ['source', 'target']
pathway = pathway.drop_duplicates()

dc.mt.ulm(data=adata_combined, net=pathway)
pathway_scores = dc.pp.get_obsm(adata=adata_combined, key="score_ulm")
sc.tl.rank_genes_groups(adata=pathway_scores, groupby="sample")

pathway_score_df = []
for i in range(len(pathway_scores.uns['rank_genes_groups']['logfoldchanges'])):
    for j in range(len(pathway_scores.uns['rank_genes_groups']['logfoldchanges'][i])):
        if (pathway_scores.uns['rank_genes_groups']['logfoldchanges'][i][j] > 0.25) and pathway_scores.uns['rank_genes_groups']['pvals_adj'][i][j] < 0.05:
            pathway_score_df.append({'group':pathway_scores.uns['rank_genes_groups']['names'].dtype.names[j], 'name':pathway_scores.uns['rank_genes_groups']['names'][i][j], 'logfoldchange':pathway_scores.uns['rank_genes_groups']['logfoldchanges'][i][j], 'pval':pathway_scores.uns['rank_genes_groups']['pvals'][i][j], 'pvals_adj':pathway_scores.uns['rank_genes_groups']['pvals_adj'][i][j]})

pathway_score_df = pd.DataFrame(pathway_score_df).sort_values(by='group')
select_top_n = 20
de_count = {x:0 for SAMPLES in datasets.values() for x in SAMPLES}
decnp_count = {x:0 for SAMPLES in datasets.values() for x in SAMPLES}

for i,row in pathway_score_df.iterrows():
    targets = pathway.loc[pathway['source']==row['name']]['target'].tolist()
    cnp_genes = list(set(targets).intersection(set(sample_DECNP_genes[row['group']])))
    if len(cnp_genes) > 0:
        if decnp_count[row['group']] < select_top_n:
            if row['group'] not in all_DE_pathways['cnp'].keys():
                all_DE_pathways['cnp'][row['group']] = {}
            all_DE_pathways['cnp'][row['group']][row['name']] = {'targets':targets, 'cnp_targets':cnp_genes}
            decnp_count[row['group']]+=1
    elif de_count[row['group']] < select_top_n:
        if row['group'] not in all_DE_pathways['non_cnp'].keys():
            all_DE_pathways['non_cnp'][row['group']] = {}
        all_DE_pathways['non_cnp'][row['group']][row['name']] = {'targets':targets}
        de_count[row['group']]+=1


In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata=pathway_scores, groupby='sample', n_genes=10, min_logfoldchange=0.25, standard_scale='var')

In [ ]:
import copy

'''
def merge_dicts_with_DE_suffix(dict1, dict2):
    # Deep copy so we don't modify original dict2
    merged = copy.deepcopy(dict2)

    for sample, categories in dict1.items():

        # Ensure sample exists in merged
        if sample not in merged:
            merged[sample] = {}

        for cat, tf_dict in categories.items():
            for tf, value in tf_dict.items():

                new_key = f"{tf}_DE"

                # If TF exists in dict2 → rename existing key
                if tf in merged[sample]:
                    merged[sample][new_key] = merged[sample].pop(tf)
                else:
                    # Add new TF from dict1
                    merged[sample][new_key] = value

    return merged
'''

def merge_tf_targets(dict1, dict2):
    dict3 = {}

    # All samples from both dictionaries
    all_samples = set(dict1['cnp'].keys()) | set(dict1['non_cnp'].keys()) | set(dict2['cnp'].keys()) | set(dict2['non_cnp'].keys())

    for cnp_type in ['cnp', 'non_cnp']:
        for sample in all_samples:
            if sample not in dict3.keys():
                dict3[sample] = {}

            # ---- Add from dict2 (already flat) ----
            if sample in dict2[cnp_type]:
                for tf, tf_data in dict2[cnp_type][sample].items():
                    dict3[sample][tf] = tf_data["targets"]

            # ---- Add from dict1 (nested under categories) ----
            if sample in dict1[cnp_type]:
                for cat in dict1[cnp_type][sample].values():
                    for tf, tf_data in cat.items():
                        dict3[sample][tf] = tf_data["targets"]

    return dict3

def select_first_n_tfs(merged_dict, n):
    selected = {}

    for sample, tf_dict in merged_dict.items():
        selected[sample] = {}

        # Separate keys
        non_de = [k for k in tf_dict.keys() if not k.endswith("_DE")]
        de = [k for k in tf_dict.keys() if k.endswith("_DE")]

        # Take first n of each
        chosen_keys = non_de[:n] + de[:n]

        # Build new dictionary
        for key in chosen_keys:
            selected[sample][key] = tf_dict[key]

    return selected

sample_clone_DE_pathways = merge_tf_targets(DE_pathways, all_DE_pathways)
#selected_regulons = select_first_n_tfs(sample_clone_DE_regulons, 8)

In [ ]:
pathway_map = pd.read_csv(DATA_ROOT + '/WP_KEGG_pathways.csv')
pathway_map = pathway_map[['pathway', 'pathway_id']].drop_duplicates()
pathway_map = dict(zip(pathway_map['pathway_id'], pathway_map['pathway']))

In [ ]:
#sc.pl.matrixplot(regulon_score, {k:[x.replace('_DE', '') for x in v.keys()] for k,v in selected_regulons.items()}, groupby='sample_clone', standard_scale='var')
pathway_keys = {k:[x for x in v.keys()] for k,v in sample_clone_DE_pathways.items()}
mp = sc.pl.matrixplot(pathway_scores, pathway_keys, groupby='sample_clone', standard_scale='var', swap_axes=True, return_fig=True)
mp_df = mp.values_df.T.copy()

cat_colormap = {#'sample':{'MGH102':'#006400', 'MGH105':'#ff0000', 'MGH122':'#ffd700', 'MGH125':'#c71585', 'MGH136':'#0F00FF', 'MGH152':'#1e90ff', 'SF10022':  '#00CED1', 'SF11979sn':'#FF8C00', 'SF4297':'#8A2BE2', 'SF6996':'#2E8B57', 'SF9259R':'#DC143C'}, 
                'sample':{'CID4495':'#006400', 'CID4513':'#ff0000', 'CID4515':'#ffd700', 'CID4523':'#c71585', 'CID44971':'#0F00FF', 'CID44991':'#1e90ff', 'Triple_negative_0126':'#00CED1', 'Triple_negative_BRCA1_0131':'#FF8C00', 'Triple_negative_BRCA1_0554':'#8A2BE2', 'TNBC1':'#2E8B57', 'TNBC2':'#DC143C', 'TNBC3':'#ffa500'},
                'clone':{'color':'#b03060','cnp':'#ffa500', 'non_cnp':'#008080'}, 
                'non_clone':{'color':'#ffdab9','cnp':'#ffa500', 'non_cnp':'#008080'}, 
                'both':{
                    'color':'#6495ed',
                    'cnp_clone':{'cnp_non_clone':'#9c1663', 'non_cnp_non_clone':'#7fa200'}, 
                    'non_cnp_clone':{'cnp_non_clone':'#ff3b83', 'non_cnp_non_clone':'#f65900'}
                    }
                }

pathway_color = {'sample':[], 'clone':[], 'cnp':[]}
for k,v in pathway_keys.items():
    for tf in v:
        pathway_color['sample'].append(cat_colormap['sample'][k])
        presence = {'clone':{'cnp':False, 'non_cnp':False}, 'sample':{'cnp':False, 'non_cnp':False}}
        for cnp_type in ['cnp', 'non_cnp']:
            if k in DE_pathways[cnp_type].keys():
                for c in DE_pathways[cnp_type][k].keys():
                    if tf in DE_pathways[cnp_type][k][c].keys():
                        presence['clone'][cnp_type] = True
            if k in all_DE_pathways[cnp_type].keys():
                if tf in all_DE_pathways[cnp_type][k].keys():
                    presence['sample'][cnp_type] = True
        total_true = sum(val for subdict in presence.values() for val in subdict.values())
        clone_true = sum(val for val in presence['clone'].values())
        sample_true = sum(val for val in presence['sample'].values())
        if total_true == 2 and clone_true == 1 and sample_true == 1:
            pathway_color['clone'].append(cat_colormap['both']['color'])
            if presence['clone']['cnp'] and presence['sample']['cnp']:
                pathway_color['cnp'].append(cat_colormap['both']['cnp_clone']['cnp_non_clone'])
            elif presence['clone']['cnp'] and presence['sample']['non_cnp']:
                pathway_color['cnp'].append(cat_colormap['both']['cnp_clone']['non_cnp_non_clone'])
            elif presence['clone']['non_cnp'] and presence['sample']['cnp']:
                pathway_color['cnp'].append(cat_colormap['both']['non_cnp_clone']['cnp_non_clone'])
            elif presence['clone']['non_cnp'] and presence['sample']['non_cnp']:
                pathway_color['cnp'].append(cat_colormap['both']['non_cnp_clone']['non_cnp_non_clone'])
            else:
                print('NON EXISTENT SCENARIO:',tf,k,presence)
        elif total_true < 2 and sample_true == 1:
            pathway_color['clone'].append(cat_colormap['non_clone']['color'])
            if presence['sample']['cnp']:
                pathway_color['cnp'].append(cat_colormap['non_clone']['cnp'])
            elif presence['sample']['non_cnp']:
                pathway_color['cnp'].append(cat_colormap['non_clone']['non_cnp'])
            else:
                print('NON EXISTENT SCENARIO:',tf,k,presence)
        elif total_true < 2 and clone_true == 1:
            pathway_color['clone'].append(cat_colormap['clone']['color'])
            if presence['clone']['cnp']:
                pathway_color['cnp'].append(cat_colormap['clone']['cnp'])
            elif presence['clone']['non_cnp']:
                pathway_color['cnp'].append(cat_colormap['clone']['non_cnp'])
            else:
                print('NON EXISTENT SCENARIO:',tf,k,presence)
        elif total_true == 2 and clone_true == 2:
            pathway_color['clone'].append(cat_colormap['clone']['color'])
            if presence['clone']['cnp']:
                pathway_color['cnp'].append(cat_colormap['clone']['cnp'])
            else:
                print('NON EXISTENT SCENARIO:',tf,k,presence)
        else:
            print('NON EXISTENT SCENARIO:',tf,k,presence)
            
pathway_color = pd.DataFrame(pathway_color)
pathway_color.index = mp_df.index
new_col_order = []
for k in pathway_keys.keys():
    for col in mp_df.columns:
        if col[0:-1] == k:
            new_col_order.append(col)

sample_color = [cat_colormap['sample'][x[0:-1]] for x in new_col_order]

g = sns.clustermap(mp_df[new_col_order], row_colors=pathway_color, col_colors=sample_color, row_cluster=False, col_cluster=False, cmap='viridis', xticklabels=True, yticklabels=True, figsize=(12,160))
g.figure.savefig(DATA_ROOT + "/manuscript_figs/pathways_clustermap_TNBC.pdf", format="pdf", dpi=300)
plt.close(g.figure)

In [ ]:
DE_pathways['cnp']['CID44971']['1'].keys()

In [ ]:
#datasets = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
#           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}
datasets = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 
            'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}

pathway = pd.read_csv(DATA_ROOT + '/WP_KEGG_pathways.csv')
associated_pathways = {}
#path_map = dict(zip(pathway['pathway_id'], pathway['pathway']))

for cnp_type in ['cnp', 'non_cnp']:
    associated_pathways[cnp_type] = {}
    for specimen, SAMPLES in datasets.items():
        for SAMPLE in SAMPLES:
            associated_pathways[cnp_type][SAMPLE] = {}
            for clone, tfs in DE_regulons[cnp_type][SAMPLE].items():
                if clone in DE_pathways[cnp_type][SAMPLE].keys():
                    paths = list(DE_pathways[cnp_type][SAMPLE][clone].keys())
                    associated_pathways[cnp_type][SAMPLE][clone] = {}
                    for tf in tfs:
                        geneset = list(set(collectri.loc[collectri['source']==tf]['target'].tolist()+[tf]))
                        gp = GProfiler(return_dataframe=True)
                        enrichment = gp.profile(organism='hsapiens', query=geneset, no_evidences=False)
                        associated_pathways[cnp_type][SAMPLE][clone][tf] = enrichment.loc[(enrichment['native'].isin(paths))].to_dict('records')
                        if len(associated_pathways[cnp_type][SAMPLE][clone][tf]) == 0:
                            del associated_pathways[cnp_type][SAMPLE][clone][tf]


In [ ]:
associated_pathways_df = {}
for cnp_type in associated_pathways.keys():
    for SAMPLE in associated_pathways[cnp_type].keys():
        for clone, tfs in associated_pathways[cnp_type][SAMPLE].items():
            for tf in tfs:
                associated_pathways_df[cnp_type+'::'+SAMPLE+'::'+str(clone)+'::'+tf] = []
                for path in associated_pathways[cnp_type][SAMPLE][clone][tf]:
                    associated_pathways_df[cnp_type+'::'+SAMPLE+'::'+str(clone)+'::'+tf].append({cnp_type+'::'+SAMPLE+'::'+str(clone)+'::'+path['native']:-1*np.log10(path['p_value'])})

flattened_data = []
for group_name, items in associated_pathways_df.items():
    for item in items:
        item['group'] = group_name
        flattened_data.append(item)

associated_pathways_df = pd.DataFrame(flattened_data).groupby('group').sum().replace(0, np.nan)

cat_colormap = {#'sample':{'MGH102':'#006400', 'MGH105':'#ff0000', 'MGH122':'#ffd700', 'MGH125':'#c71585', 'MGH136':'#0F00FF', 'MGH152':'#1e90ff', 'SF10022':  '#00CED1', 'SF11979sn':'#FF8C00', 'SF4297':'#8A2BE2', 'SF6996':'#2E8B57', 'SF9259R':'#DC143C'}, 
                'sample':{'CID4495':'#006400', 'CID4513':'#ff0000', 'CID4515':'#ffd700', 'CID4523':'#c71585', 'CID44971':'#0F00FF', 'CID44991':'#1e90ff', 'Triple_negative_0126':'#00CED1', 'Triple_negative_BRCA1_0131':'#FF8C00', 'Triple_negative_BRCA1_0554':'#8A2BE2', 'TNBC1':'#2E8B57', 'TNBC2':'#DC143C', 'TNBC3':'#ffa500'},
                'cnp_type':{'cnp':'#ffa500', 'non_cnp':'#008080'}, 
                'clone':{'0':'#2E8B57', '1':'#6495ed', '2':'#9c1663', '3':'#7fa200', '4':'#ff3b83', '5':'#f65900'}
                }

row_colors = pd.DataFrame([{'cnp_type':cat_colormap['cnp_type'][x.split('::')[0]], 'sample':cat_colormap['sample'][x.split('::')[1]], 'clone':cat_colormap['clone'][x.split('::')[2]]} for x in associated_pathways_df.index])
col_colors = pd.DataFrame([{'cnp_type':cat_colormap['cnp_type'][x.split('::')[0]], 'sample':cat_colormap['sample'][x.split('::')[1]], 'clone':cat_colormap['clone'][x.split('::')[2]]} for x in associated_pathways_df.columns])

associated_pathways_df.index = [x.split('::')[3] for x in associated_pathways_df.index]
associated_pathways_df.columns = [x.split('::')[3] for x in associated_pathways_df.columns]
associated_pathways_df.columns = [pathway_map[x] for x in associated_pathways_df.columns]

row_colors.index = associated_pathways_df.index
col_colors.index = associated_pathways_df.columns

g = sns.clustermap(associated_pathways_df, row_colors=row_colors, col_colors=col_colors, row_cluster=False, col_cluster=False, cmap='plasma', xticklabels=True, yticklabels=True, figsize=(50,50))
g.figure.savefig(DATA_ROOT + "/manuscript_figs/regulon_vs_pathway_TNBC.pdf", format="pdf", dpi=300)
plt.close(g.figure)

In [ ]:
modules = pd.read_csv(DATA_ROOT + '/Glio/Glio_meta_modules.csv').to_dict(orient='list')
print(modules)

In [ ]:
temp = sc.read_h5ad(DATA_ROOT + '/Glio/Data_Neftel2019_Brain/SmartSeq2/MGH105_clone.h5ad')
temp

## Bootstrap & SCENIC analysis

In [ ]:
sns.set_theme(font_scale=1.5)
sns.set_style('white')

collectri = dc.op.collectri(organism="human")
modules = pd.read_csv(DATA_ROOT + '/Glio/Glio_meta_modules.csv').to_dict(orient='list')
module_related_genes = {}
for module, genes in modules.items():
    module_related_genes[module] = {'source':[],'target':[]}
    subnet = collectri.loc[(collectri['source'].isin(genes)) | (collectri['target'].isin(genes))]
    module_related_genes[module]['source'] = list(set(subnet['source'].tolist()))
    module_related_genes[module]['target'] = list(set(subnet['target'].tolist()))
    related_genes = list(set(subnet['source'].tolist()+subnet['target'].tolist()))

perform_bootstrap = True
correlations = {}
dfs1 = {}
dfs2 = {}

G_nx_total = {}
G_df_total = {}

datasets = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}
#datasets = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 
#            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 
#            'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}

for specimen, SAMPLES in datasets.items():
    for SAMPLE in SAMPLES:
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone.h5ad'
        else:
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'
        
        adata = sc.read_h5ad(SAMPLE_PATH)
        clones = list(adata.obs.inferred_clone.unique())
        for clone in clones:
            if clone == 'N' or (specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2' and clone == '0') or (specimen=='Wang_2019' and clone == '2') or len(adata.obs.loc[adata.obs['inferred_clone']==clone]) < 20:
                adata = adata[adata.obs.loc[adata.obs['inferred_clone']!=clone].index].copy()
        print(SAMPLE)
        print(adata.obs['inferred_clone'].unique())
        sc.pp.filter_cells(adata, min_genes=10)
        sc.pp.filter_genes(adata, min_cells=2)
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        sc.tl.pca(adata)
        sc.pp.neighbors(adata)
        sc.tl.umap(adata)
        sc.pl.umap(adata=adata, color="inferred_clone")

        #bootstrap analysis
        if perform_bootstrap:
            clones = list(adata.obs.inferred_clone.unique())
            boot_grns = {x:{} for x in clones}
            for clone in clones:
                for rate in [0.9]:
                    boot_grns[clone][rate] = []
                    for i in range(100):
                        network = pd.read_csv(f"{DATA_ROOT}/bootstrap_celloracle/{specimen.replace('/Data_Neftel2019_Brain/SmartSeq2','')}/{SAMPLE}/CL{clone}-{i+1}-celloracle-net.tsv", sep='\t')
                        G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                        boot_grns[clone][rate].append(G)
            
            all_grns = {}
            for clone in clones:
                network = pd.read_csv(f"{DATA_ROOT}/bootstrap_celloracle/{specimen.replace('/Data_Neftel2019_Brain/SmartSeq2','')}/{SAMPLE}/CL{clone}-whole-celloracle-net.tsv", sep='\t')
                G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                all_grns[clone] = G
            
            for clone in clones:
                for rate in [0.90]:
                    for idx in range(100):
                        _G = boot_grns[clone][rate][idx]
                        for e in _G.edges():
                            if all_grns[clone].has_edge(e[0], e[1]):
                                if not f"support_{rate}" in all_grns[clone].edges[e[0], e[1]].keys():
                                    all_grns[clone].edges[e[0], e[1]][f"support_{rate}"] = 1
                                else:
                                    all_grns[clone].edges[e[0], e[1]][f"support_{rate}"] += 1
            
            G_data = {clone:{"source": [], "target": [], "support_0.90": []} for clone in clones}
            for clone in clones:
                for s, t, vals in all_grns[clone].edges(data=True):
                    G_data[clone]["source"].append(s)
                    G_data[clone]["target"].append(t)
                    try:
                        G_data[clone]["support_0.90"].append(vals["support_0.9"])
                    except KeyError:
                        G_data[clone]["support_0.90"].append(0)

            G_df = {clone:pd.DataFrame(data=G_data[clone]) for clone in clones}
            G_nx = {clone:nx.from_pandas_edgelist(G_df[clone], source='source', target='target', edge_attr='support_0.90', create_using=nx.DiGraph) for clone in clones}
            G_df_total[SAMPLE] = G_df.copy()
            G_nx_total[SAMPLE] = G_nx.copy()
        
                
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            #Pathway enrichment across clones
            pathway = dc.op.hallmark(organism="human")
            #Add meta modules
            pathway = pathway.to_dict(orient='list')
            for module, genes in modules.items():
                subnet = collectri.loc[(collectri['source'].isin(genes)) | (collectri['target'].isin(genes))]
                related_genes = list(set(subnet['source'].tolist()+subnet['target'].tolist()))
                for gene in related_genes:
                    pathway['source'].append(module)
                    pathway['target'].append(gene)
            pathway = pd.DataFrame(pathway)
            dc.mt.ulm(data=adata, net=pathway)
            pathway_scores = dc.pp.get_obsm(adata=adata, key="score_ulm")
            pathway_scores_df = dc.tl.rankby_group(adata=pathway_scores, groupby="inferred_clone")
            pathway_scores_df = pathway_scores_df[pathway_scores_df["stat"] > 0]
            pathway_scores_df = pathway_scores_df.loc[pathway_scores_df['name'].isin(['MES2', 'MES1', 'AC', 'OPC', 'NPC1', 'NPC2', 'G1/S', 'G2/M'])]

            mp = sc.pl.matrixplot(
                adata=pathway_scores,
                var_names=['MES2', 'MES1', 'AC', 'OPC', 'NPC1', 'NPC2', 'G1/S', 'G2/M'],#pathway_markers,
                groupby="inferred_clone",
                dendrogram=True,#standard_scale="var",
                colorbar_title="Z-scaled scores",
                cmap="RdBu_r",
                return_fig=True
            )
            
            mp.get_axes()['mainplot_ax'].tick_params(left=True, bottom=True)
            mp.show()
            #mp.savefig(DATA_ROOT + '/Glio/Data_Neftel2019_Brain/SmartSeq2/meta_module_plots/'+SAMPLE+'_clone.png')
            
            '''
            temp = pd.DataFrame()
            temp['meta_mod'] = adata.obs[['MESlike2', 'MESlike1', 'AClike', 'OPClike', 'NPClike1', 'NPClike2', 'G1S', 'G2M']].astype(float).idxmax(axis=1)
            temp['clones'] = adata.obs['inferred_clone']
            
            cross_tab = pd.crosstab(temp['clones'], temp['meta_mod'])
            plt.figure(figsize=(4, 2))
            ax = sns.heatmap(cross_tab, cmap='RdBu_r', square=True)
            ax.tick_params(left=True, bottom=True)
            
            temp1 = temp.groupby('clones')['meta_mod'].value_counts().unstack(level=1)
            temp1 = temp1.rename(columns={'MESlike2':'MES2', 'MESlike1':'MES1', 'AClike':'AC', 'OPClike':'OPC', 'NPClike1':'NPC1', 'NPClike2':'NPC2', 'G1S':'G1/S', 'G2M':'G2/M'})
            temp1 = temp1.T
            temp2 = mp.values_df.T
            dfs1[SAMPLE] = temp1.copy()
            dfs2[SAMPLE] = temp2.copy()
            correlations[SAMPLE] = {}
            '''
            temp = adata.obs[['MESlike2', 'MESlike1', 'AClike', 'OPClike', 'NPClike1', 'NPClike2', 'G1S', 'G2M']].copy()
            temp['clones'] = adata.obs['inferred_clone']
            temp1 = temp.copy()
            temp1 = temp1.rename(columns={'MESlike2':'MES2', 'MESlike1':'MES1', 'AClike':'AC', 'OPClike':'OPC', 'NPClike1':'NPC1', 'NPClike2':'NPC2', 'G1S':'G1/S', 'G2M':'G2/M'})
            temp2 = pathway_scores.to_df()[['MES2', 'MES1', 'AC', 'OPC', 'NPC1', 'NPC2', 'G1/S', 'G2/M']].copy()
            #data for correlation matrix
            correlations[SAMPLE] = {'temp1':temp1.copy(), 'temp2':temp2.copy()}
            #for col in temp1.columns:
            #    indices = temp1[temp1[col].notna()].index
            #    correlations[SAMPLE][col] = temp1.loc[indices, col].corr(temp2.loc[indices, col])#temp1.loc[indices, col].rank().corr(temp2.loc[indices, col].rank())
        

print(correlations)


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from itertools import product

def compute_cross_sample_correlations(data: dict) -> dict:
    """
    Computes Spearman rank correlation matrices between samples.

    For each ordered pair (sample_A, sample_B):
      - Groups temp1 of sample_A by 'clones', averages all non-clone columns  → shape (n_clones_A, n_features)
      - Groups temp2 of sample_B by the clone labels from temp1 of sample_B, averages all columns → shape (n_clones_B, n_features)
      - Computes a (n_clones_A x n_clones_B) Spearman correlation matrix where
        entry [i, j] = Spearman correlation between the feature vector of clone i (from sample_A temp1)
                       and the feature vector of clone j (from sample_B temp2)

    Returns:
        correlation[sample_A][sample_B] = pd.DataFrame of shape (n_clones_A, n_clones_B)
    """

    # Step 1: Pre-compute clone-averaged matrices for all samples
    # For each sample, compute:
    #   - averaged_temp1: temp1 grouped by 'clones', mean of all other columns
    #   - averaged_temp2: temp2 grouped by clone labels taken from temp1, mean of all columns
    averaged = {}
    for sample, temps in data.items():
        temp1 = temps['temp1'].astype(float)
        temp2 = temps['temp2']

        # Clone labels come from temp1 of this sample
        clone_labels = temp1['clones']
        feature_cols = [c for c in temp1.columns if c != 'clones']

        # Average temp1 (non-clone columns) grouped by clones
        avg_t1 = temp1[feature_cols].groupby(clone_labels).mean()
        # avg_t1.index = clone values, columns = feature columns

        # Average temp2 (all columns) grouped by the same clone labels
        avg_t2 = temp2.groupby(clone_labels).mean()

        averaged[sample] = {
            'temp1': avg_t1,  # shape: (n_clones, n_features)
            'temp2': avg_t2,  # shape: (n_clones, n_features)
        }
    
    #print(averaged)

    # Step 2: For every ordered pair (A, B), correlate averaged temp1 of A with averaged temp2 of B
    correlation = {s: {} for s in data}

    for sample_a, sample_b in product(data.keys(), repeat=2):
        avg_t1_a = averaged[sample_a]['temp1']  # rows = clones of A, cols = features
        avg_t2_b = averaged[sample_b]['temp2']  # rows = clones of B, cols = features

        clones_a = avg_t1_a.index.tolist()
        clones_b = avg_t2_b.index.tolist()

        # Align on shared feature columns
        shared_cols = avg_t1_a.columns.intersection(avg_t2_b.columns)
        t1_vals = avg_t1_a[shared_cols].values  # shape: (n_clones_A, n_features)
        t2_vals = avg_t2_b[shared_cols].values  # shape: (n_clones_B, n_features)

        # Build correlation matrix: entry [i, j] = Spearman corr between
        # feature vector of clone i (from A's temp1) and clone j (from B's temp2)
        corr_matrix = np.full((len(clones_a), len(clones_b)), np.nan)

        for i, vec_a in enumerate(t1_vals):
            for j, vec_b in enumerate(t2_vals):
                # spearmanr returns (correlation, pvalue); we take only the correlation
                #print(vec_a, vec_b)
                rho, _ = spearmanr(vec_a, vec_b)
                corr_matrix[i, j] = rho

        correlation[sample_a][sample_b] = pd.DataFrame(
            corr_matrix,
            index=clones_a,
            columns=clones_b
        )

    return correlation

corr_mat = compute_cross_sample_correlations(correlations)

In [ ]:
corr_mat['MGH102']['MGH105']

In [ ]:
corr_mat['MGH102']['MGH105']

In [ ]:
correlations['MGH105']['temp1']

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def maximize_assignment(df):
    matrix = df.values
    n, m = matrix.shape

    # linear_sum_assignment minimizes by default, so negate to maximize
    row_ind, col_ind = linear_sum_assignment(-matrix)

    # Build result
    assignments = []
    total = 0
    for r, c in zip(row_ind, col_ind):
        val = matrix[r, c]
        assignments.append({
            "row": df.index[r],
            "col": df.columns[c],
            "value": val
        })
        total += val

    result_df = pd.DataFrame(assignments)
    #print(f"Total score: {total:.4f}")
    #print(result_df)
    return result_df


corr_mat = {}

for key1 in correlations.keys():
    corr_mat[key1] = {}
    for key2 in correlations.keys():
        temp = pd.concat([correlations[key1]['temp1'].fillna(0), correlations[key2]['temp2'].fillna(0)], axis=1, keys=['df1', 'df2']).corr(method='spearman').loc['df2', 'df1']
        if key1==key2:
            corr_mat[key1][key2] = np.diag(temp).mean()
        else:
            corr_mat[key1][key2] = maximize_assignment(temp)['value'].mean()


In [ ]:
temp = pd.concat([correlations['MGH102']['temp1'].fillna(0), correlations['MGH102']['temp2'].fillna(0)], axis=1, keys=['df1', 'df2']).corr(method='spearman').loc['df2', 'df1']
g = sns.heatmap(temp, cmap='RdBu_r')
g.set(ylabel='Meta-program distribution', xlabel='Meta-program activity')

In [ ]:
g = sns.heatmap(pd.DataFrame(corr_mat), cmap='RdBu_r')
g.set(ylabel='Meta-program distribution', xlabel='Meta-program activity')

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame(
    np.array([[4, 1, 3],
              [2, 0, 5],
              [3, 2, 2]]),
    index=["r1", "r2", "r3"],
    columns=["c1", "c2", "c3"]
)

maximize_assignment(df)
# Total score: 13.0
# r1 -> c1 (4), r2 -> c3 (5), r3 -> c2 (2) ... wait
# Actually: r1->c1=4, r2->c3=5, r3->c2=2 = 11
# Optimal: r1->c3=3? No: r1->c1=4, r2->c3=5, r3->c2=2 = 11

In [ ]:
fig, ax = plt.subplots()
ax.boxplot([x for y in correlations.keys() for x in correlations[y].values() if not math.isnan(x)])
ax.set_ylabel("Spearmann correlation")
ax.set_title("Distribution of spearmann correlation \nbetween meta module score and GRN activity across clones in multiple datasets")
plt.show()

In [ ]:
##############################################
#########DO NOT RUN!!!!################
##############################################



datasets = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}

modules = pd.read_csv(DATA_ROOT + '/Glio/Glio_meta_modules.csv').to_dict(orient='list')

perform_bootstrap = True
correlations = {}

G_nx_total = {}

for specimen, SAMPLES in datasets.items():
    for SAMPLE in SAMPLES:
        if specimen != 'Wang_2019':
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone.h5ad'
        else:
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'
        adata = sc.read_h5ad(SAMPLE_PATH)
        adata = adata[adata.obs.loc[adata.obs['inferred_clone']!='N'].index].copy()
        if specimen!='Wang_2019':
            adata = adata[adata.obs.loc[adata.obs['inferred_clone']!='0'].index].copy()
        else:
            adata = adata[adata.obs.loc[adata.obs['inferred_clone']!='2'].index].copy()
        print(adata.obs['inferred_clone'].unique())
        sc.pp.filter_cells(adata, min_genes=10)
        sc.pp.filter_genes(adata, min_cells=2)
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        sc.tl.pca(adata)
        sc.pp.neighbors(adata)
        sc.tl.umap(adata)
        sc.pl.umap(adata=adata, color="inferred_clone")

        #bootstrap analysis
        if perform_bootstrap:
            clones = list(adata.obs.inferred_clone.unique())
            boot_grns = {x:{} for x in clones}
            for clone in tqdm(clones):
                for rate in [0.9]:
                    boot_grns[clone][rate] = []
                    for i in range(1, 101):
                        network = pd.read_csv(f"{DATA_ROOT}/bootstrap_celloracle/{specimen.replace('/Data_Neftel2019_Brain/SmartSeq2','')}/{SAMPLE}/CL{clone}-{i}-celloracle-net.tsv", sep='\t')
                        G1 = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                        #network = pd.read_csv(f"{DATA_ROOT}/bootstrap_grnboost/Glio/{SAMPLE}/CL{clone}-{i}-grnboost-net.tsv", sep='\t')
                        #G2 = nx.from_pandas_edgelist(network.loc[network.importance>0], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                        G = G1.copy()#nx.intersection(G1, G2)
                        boot_grns[clone][rate].append(G)
            
            all_grns = {}
            for clone in tqdm(clones):
                network = pd.read_csv(f"{DATA_ROOT}/bootstrap_celloracle/{specimen.replace('/Data_Neftel2019_Brain/SmartSeq2','')}/{SAMPLE}/CL{clone}-whole-celloracle-net.tsv", sep='\t')
                G1 = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                #network = pd.read_csv(f"{DATA_ROOT}/bootstrap_grnboost/Glio/{SAMPLE}/CL{clone}-whole-grnboost-net.tsv", sep='\t')
                #G2 = nx.from_pandas_edgelist(network.loc[network.importance>0], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                G = G1.copy()#nx.intersection(G1, G2)
                all_grns[clone] = G
            
            for clone in tqdm(clones):
                for rate in [0.90]:
                    for idx in range(100):
                        _G = boot_grns[clone][rate][idx]
                        for e in _G.edges():
                            if all_grns[clone].has_edge(e[0], e[1]):
                                if not f"support_{rate}" in all_grns[clone].edges[e[0], e[1]].keys():
                                    all_grns[clone].edges[e[0], e[1]][f"support_{rate}"] = 1
                                else:
                                    all_grns[clone].edges[e[0], e[1]][f"support_{rate}"] += 1
            
            G_data = {clone:{"source": [], "target": [], "support_0.90": []} for clone in clones}
            for clone in tqdm(clones):
                for s, t, vals in all_grns[clone].edges(data=True):
                    G_data[clone]["source"].append(s)
                    G_data[clone]["target"].append(t)
                    try:
                        G_data[clone]["support_0.90"].append(vals["support_0.9"])
                    except KeyError:
                        G_data[clone]["support_0.90"].append(0)

            G_df = {clone:pd.DataFrame(data=G_data[clone]) for clone in clones}
            G_nx = {clone:nx.from_pandas_edgelist(G_df[clone], source='source', target='target', edge_attr='support_0.90', create_using=nx.DiGraph) for clone in clones}
            G_nx_total[SAMPLE] = G_nx.copy()
        
        #uncomment for correlation between clone distribution and GRN clone activity distribution of meta modules
        '''
        #Pathway enrichment across clones
        pathway = dc.op.hallmark(organism="human")
        #Add meta modules
        pathway = pathway.to_dict(orient='list')
        for module, genes in modules.items():
            subnet = collectri.loc[(collectri['source'].isin(genes)) | (collectri['target'].isin(genes))]
            related_genes = list(set(subnet['source'].tolist()+subnet['target'].tolist()))
            for gene in related_genes:
                pathway['source'].append(module)
                pathway['target'].append(gene)
        pathway = pd.DataFrame(pathway)
        dc.mt.ulm(data=adata, net=pathway)
        pathway_scores = dc.pp.get_obsm(adata=adata, key="score_ulm")
        pathway_scores_df = dc.tl.rankby_group(adata=pathway_scores, groupby="inferred_clone")
        pathway_scores_df = pathway_scores_df[pathway_scores_df["stat"] > 0]
        pathway_scores_df = pathway_scores_df.loc[pathway_scores_df['name'].isin(['MES2', 'MES1', 'AC', 'OPC', 'NPC1', 'NPC2', 'G1/S', 'G2/M'])]

        mp = sc.pl.matrixplot(
            adata=pathway_scores,
            var_names=['MES2', 'MES1', 'AC', 'OPC', 'NPC1', 'NPC2', 'G1/S', 'G2/M'],#pathway_markers,
            groupby="inferred_clone",
            dendrogram=True,#standard_scale="var",
            colorbar_title="Z-scaled scores",
            cmap="RdBu_r",
            return_fig=True
        )
        
        mp.show()
        #mp.savefig(DATA_ROOT + '/Glio/Data_Neftel2019_Brain/SmartSeq2/meta_module_plots/'+SAMPLE+'_clone.png')
        
        temp = pd.DataFrame()
        temp['meta_mod'] = adata.obs[['MESlike2', 'MESlike1', 'AClike', 'OPClike', 'NPClike1', 'NPClike2', 'G1S', 'G2M']].astype(float).idxmax(axis=1)
        temp['clones'] = adata.obs['inferred_clone']
        temp1 = temp.groupby('clones')['meta_mod'].value_counts().unstack(level=1)
        temp1 = temp1.rename(columns={'MESlike2':'MES2', 'MESlike1':'MES1', 'AClike':'AC', 'OPClike':'OPC', 'NPClike1':'NPC1', 'NPClike2':'NPC2', 'G1S':'G1/S', 'G2M':'G2/M'})
        temp1 = temp1.T
        temp2 = mp.values_df.T
        correlations[SAMPLE] = {}
        for col in temp1.columns:
            indices = temp1[temp1[col].notna()].index
            correlations[SAMPLE][col] = temp1.loc[indices, col].rank().corr(temp2.loc[indices, col].rank())
        '''

print(correlations)


In [ ]:
scenic_all_grns_grnboost = {}
scenic_all_grns_celloracle = {}
scenic_union_celloracle_grn = nx.DiGraph()

datasets = {'Glio/Data_Neftel2019_Brain/SmartSeq2': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'],
           'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}
#datasets = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 
#            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 
#            'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}

for specimen, SAMPLES in datasets.items():
    for SAMPLE in SAMPLES:
        if specimen=='Glio/Data_Neftel2019_Brain/SmartSeq2':
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone.h5ad'
        else:
            SAMPLE_PATH = DATA_ROOT + '/'+specimen+'/'+SAMPLE+'_clone_correct.h5ad'

        scenic_all_grns_grnboost[SAMPLE] = {}
        scenic_all_grns_celloracle[SAMPLE] = {}
        adata = sc.read_h5ad(SAMPLE_PATH)
        #adata = adata[adata.obs.loc[adata.obs['inferred_clone']!='N'].index].copy()
        #adata = adata[adata.obs.loc[adata.obs['inferred_clone']!='0'].index].copy()
        print(adata.obs['inferred_clone'].unique())
        sc.pp.filter_cells(adata, min_genes=10)
        sc.pp.filter_genes(adata, min_cells=2)
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        sc.tl.pca(adata)
        sc.pp.neighbors(adata)
        sc.tl.umap(adata)
        clones = list(adata.obs.inferred_clone.unique())
        for clone in clones:
            if len(adata.obs.loc[adata.obs['inferred_clone']==clone]) > 20:
                #network = pd.read_csv(f"{DATA_ROOT}/bootstrap_grnboost/{specimen.replace('/Data_Neftel2019_Brain/SmartSeq2','')}/{SAMPLE}/CL{clone}-whole-scenic-net.tsv", sep='\t')
                #G = nx.from_pandas_edgelist(network, source='Gene1', target='Gene2', edge_attr='weight', create_using=nx.DiGraph)
                #scenic_all_grns_grnboost[SAMPLE][clone] = G.copy()
                network = pd.read_csv(f"{DATA_ROOT}/bootstrap_celloracle/{specimen.replace('/Data_Neftel2019_Brain/SmartSeq2','')}/{SAMPLE}/CL{clone}-whole-scenic-net.tsv", sep='\t')
                G = nx.from_pandas_edgelist(network, source='Gene1', target='Gene2', edge_attr='weight', create_using=nx.DiGraph)
                scenic_all_grns_celloracle[SAMPLE][clone] = G.copy()
                scenic_union_celloracle_grn = nx.compose(scenic_union_celloracle_grn, G)
    

In [ ]:
import matplotlib.pyplot as plt

def plot_sample_grid(samples_dict, modules, edge_attr="support_0.90"):
    """
    For each sample produces a grid:
        rows = modules
        cols = categories
    Now shows module name at the beginning of each row.
    """

    for sample, cat_dict in samples_dict.items():

        categories = list(cat_dict.keys())
        module_names = list(modules.keys())

        n_rows = len(module_names)
        n_cols = len(categories)

        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(4 * n_cols, 3.5 * n_rows),
            squeeze=False
        )

        fig.suptitle(f"Sample: {sample}", fontsize=20)

        # Iterate over modules × categories
        for i, module_name in enumerate(module_names):
            module_nodes = modules[module_name]

            # --- NEW: Module label on the left margin ---
            fig.text(
                0.02,                               # x-position (left margin)
                (n_rows - i - 0.5) / n_rows,        # y-position (center of row)
                module_name,
                va='center', ha='left',
                fontsize=14
            )

            for j, cat in enumerate(categories):
                ax = axes[i][j]
                G = cat_dict[cat]

                # Extract valid nodes for the module that exist in the graph
                valid_nodes = [n for n in module_nodes if n in G.nodes()]
                subG = G.subgraph(valid_nodes).copy()

                if i == 0:
                    ax.set_title(f"{cat}", fontsize=14)

                if subG.number_of_nodes() == 0:
                    ax.text(0.5, 0.5, "No nodes", ha='center', va='center')
                    ax.axis('off')
                    continue

                # Layout
                pos = nx.spring_layout(subG, seed=42)

                # Edge attributes
                edge_vals = [subG[u][v].get(edge_attr) for u, v in subG.edges()]
                has_attr = any(val is not None for val in edge_vals)

                # Draw nodes
                nx.draw_networkx_nodes(
                    subG, pos,
                    node_size=250,
                    node_color="lightblue",
                    ax=ax
                )

                # Draw edges with or without attribute
                if has_attr:
                    nx.draw_networkx_edges(
                        subG, pos,
                        edge_color=edge_vals,
                        edge_cmap=plt.cm.viridis,
                        width=2,
                        ax=ax
                    )
                else:
                    nx.draw_networkx_edges(
                        subG, pos,
                        width=2,
                        ax=ax
                    )

                # Draw labels
                nx.draw_networkx_labels(subG, pos, font_size=9, ax=ax)
                ax.set_axis_off()

        plt.tight_layout(rect=[0.06, 0, 1, 0.95])  # leave room for row labels
        plt.show()


In [ ]:
#plot_sample_grid({key:G_nx_total[key] for key in ['MGH']}, modules=modules)
plot_sample_grid({key:scenic_all_grns_grnboost[key] for key in ['MGH105']}, modules=modules)

In [ ]:
plot_sample_grid({key:scenic_all_grns_celloracle[key] for key in ['MGH125']}, modules=modules)

In [ ]:
G_df_meta = {}
for sample, val in G_df_total.items():
    G_df_meta[sample] = {}
    for clone, df in val.items():
        G_df_meta[sample][clone] = {}
        for module, module_genes in modules.items():
            G_df_meta[sample][clone][module] = G_df_total[sample][clone].loc[(G_df_total[sample][clone]['source'].isin(module_genes)) & (G_df_total[sample][clone]['target'].isin(module_genes))].copy()

In [ ]:
#sample_list_color = {'CID4495':'#1F77B4', 'CID4513':'#FF7F0E', 'CID4515':'#279E68', 'CID4523':'#D62728', 'CID44971':'#AA40FC', 'CID44991':'#8C564B', 'TNBC1':'#E16EBE', 'TNBC2':'#B5BD61', 'TNBC3':'#17BECF', 'Triple_negative_0126':'#AEC7E8', 'Triple_negative_BRCA1_0131':'#FFBB78', 'Triple_negative_BRCA1_0554':'#98DF8A'}
sample_list_color = {'MGH102':'#1F77B4', 'MGH105':'#FF7F0E', 'MGH122':'#279E68', 'MGH125':'#D62728', 'MGH136':'#AA40FC', 'MGH152':'#8C564B', 'SF10022':'#E16EBE', 'SF11979sn':'#B5BD61', 'SF4297':'#17BECF', 'SF6996':'#AEC7E8', 'SF9259R':'#FFBB78'}

scenic_union_celloracle_module_grn = {}
for module, module_genes in modules.items():
    scenic_union_celloracle_module_grn[module] = scenic_union_celloracle_grn.subgraph(module_genes)\

scenic_module_bootstrap_df = {}
for module, module_genes in modules.items():
    scenic_module_bootstrap_df[module] = {}
    edgeset = [x[0]+':'+x[1] for x in scenic_union_celloracle_module_grn[module].edges()]
    for edge in edgeset:
        scenic_module_bootstrap_df[module][edge] = {}
    for sample, val in scenic_all_grns_celloracle.items():
        for clone, grn in val.items():
            if (('MGH' in sample and clone!='0') or (('MGH' not in sample) and clone!='2')) and clone!='N':
                subgraph = scenic_all_grns_celloracle[sample][clone].subgraph(module_genes)
                for edge in subgraph.edges():
                    source = edge[0]
                    target = edge[1]
                    bootstrap_val = G_df_meta[sample][clone][module].loc[(G_df_meta[sample][clone][module]['source']==source) & (G_df_meta[sample][clone][module]['target']==target)]['support_0.90'].tolist()
                    if len(bootstrap_val) > 1:
                        print('ERROR: Bootstrap issue', bootstrap_val, edge, module, sample, clone)
                        continue
                    if len(bootstrap_val) == 0:
                        scenic_module_bootstrap_df[module][edge[0]+':'+edge[1]][sample+'-'+clone] = 0
                    else:    
                        scenic_module_bootstrap_df[module][edge[0]+':'+edge[1]][sample+'-'+clone] = bootstrap_val[0]
                        
scenic_module_bootstrap_df = {module:pd.DataFrame(df) for module, df in scenic_module_bootstrap_df.items()}
scenic_module_bootstrap_adata = {}
for module, df in scenic_module_bootstrap_df.items():
    if len(df)>0:
        temp = sc.AnnData(df)
        temp.obs['sample'] = [x.split('-')[0] for x in temp.obs_names]
        temp.obs['sample'] = temp.obs['sample'].astype('category')
        temp.uns['sample_colors'] = [sample_list_color[x] for x in temp.obs['sample'].cat.categories]
        scenic_module_bootstrap_adata[module] = temp.copy()

In [ ]:
scenic_module_bootstrap_adata

In [ ]:

for module in scenic_module_bootstrap_adata.keys():
    print(module)
    sc.pl.heatmap(scenic_module_bootstrap_adata[module], var_names=scenic_module_bootstrap_adata[module].var_names, groupby='sample', show_gene_labels=True, cmap='viridis')

In [ ]:
boot_grns = pkl.load(open(DATA_ROOT + '/Glio/bootstrap_whole.pkl','rb'))
boot_grns_df = {}
for clone in boot_grns['MGH'].keys():
    boot_grns_df[clone] = nx.to_pandas_edgelist(boot_grns['MGH'][clone])
    boot_grns_df[clone] = boot_grns_df[clone].loc[boot_grns_df[clone]['support_0.90']!=0]

In [ ]:
boot_grns = G_nx_total.copy()
boot_grns_df = {}
for clone in boot_grns['MGH105'].keys():
    boot_grns_df[clone] = nx.to_pandas_edgelist(boot_grns['MGH105'][clone])
    boot_grns_df[clone] = boot_grns_df[clone].loc[boot_grns_df[clone]['support_0.90']!=0]

In [ ]:
np.histogram(boot_grns_df['1']['support_0.90'])

In [ ]:
clone1 = '1'
clone2 = '2'

G = nx.intersection_all([boot_grns['MGH105'][x] for x in [clone1,clone2]])
temp = pd.merge(boot_grns_df[clone1], nx.to_pandas_edgelist(G), how='inner', on=['source', 'target'])
temp = pd.merge(temp, boot_grns_df[clone2], how='inner', on=['source', 'target'])

In [ ]:
temp

In [ ]:
for i,row in temp.iterrows():
    source_pos = []
    target_pos = []
    if row['source'] in DECNP_genes[clone1]:
        source_pos.append(DECNP_genes[clone1].index(row['source']))
    if row['source'] in DECNP_genes[clone2]:
        source_pos.append(DECNP_genes[clone2].index(row['source']))
    if row['target'] in DECNP_genes[clone1]:
        target_pos.append(DECNP_genes[clone1].index(row['target']))
    if row['target'] in DECNP_genes[clone2]:
        target_pos.append(DECNP_genes[clone2].index(row['target']))
    if abs(row['support_0.90_x'] - row['support_0.90_y']) > 20 and (len(source_pos) > 0 or len(target_pos) > 0) and (np.mean(source_pos) < 300 or np.mean(target_pos) < 300):
        print(row['source'], row['target'], row['support_0.90_x'], row['support_0.90_y'], source_pos, target_pos)

# Centrality

In [ ]:
import os

def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        if '-whole-scenic' in entry:
            full_path = os.path.join(directory_path, entry)
            if os.path.isfile(full_path):
                clones.append(entry.split('-')[0][2])
    return list(set(clones))

all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'], 'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554'], 'Gao_2021':['TNBC1', 'TNBC2', 'TNBC3']}
#all_data = {'Glio': ['MGH105', 'MGH102', 'MGH122', 'MGH125', 'MGH136', 'MGH152'], 'Wang_2019':['SF10022', 'SF11979sn', 'SF4297', 'SF6996', 'SF9259R']}

total_G_df = {}
total_all_grns = {}

for paper, datasets in all_data.items():
    for dataset in tqdm(datasets):
        methods=['celloracle']

        SAMPLE_PATHS = {method:DATA_ROOT + "/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = dataset
        print('sample',SAMPLE)

        clones = get_clones(SAMPLE_PATHS['celloracle'])

        '''
        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()
        '''
        print('clones',clones)

        boot_grns = {x:{} for x in methods}

        for method in methods:
            boot_grns[method] = {x:{} for x in clones}
            for clone in clones:
                for rate in [0.9]:
                    boot_grns[method][clone][rate] = []
                    for i in range(1, 101):
                        network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                        if method=='celloracle':
                            G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                        if method=='grnboost':
                            thresh = np.histogram(network['importance'])[1][1]
                            G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                        boot_grns[method][clone][rate].append(G)

        all_grns = {x:{} for x in methods+['scenic']}

        for method in methods+['scenic']:
            for clone in clones:
                if method=='celloracle':
                    network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-whole-"+method+"-net.tsv", sep='\t')
                    G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                #if method=='grnboost':
                #    thresh = np.histogram(network['importance'])[1][1]
                #    G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                if method=='scenic':
                    network = pd.read_csv(f"{DATA_ROOT}/bootstrap_celloracle/{paper}/{dataset}/CL{clone}-whole-scenic-net.tsv", sep='\t')
                    G = nx.from_pandas_edgelist(network, source='Gene1', target='Gene2', edge_attr='weight', create_using=nx.DiGraph)
                all_grns[method][clone] = G
                
        for method in methods+['scenic']:
            for clone in clones:
                for rate in [0.90]:
                    for idx in range(100):
                        _G = boot_grns['celloracle'][clone][rate][idx]
                        for e in _G.edges():
                            if all_grns[method][clone].has_edge(e[0], e[1]):
                                if not f"support_{rate}" in all_grns[method][clone].edges[e[0], e[1]].keys():
                                    all_grns[method][clone].edges[e[0], e[1]][f"support_{rate}"] = 1
                                else:
                                    all_grns[method][clone].edges[e[0], e[1]][f"support_{rate}"] += 1

        G_data = {method:{clone:{"source": [], "target": [], "support_0.90": []} for clone in clones} for method in methods+['scenic']}
        for method in methods+['scenic']:
            for clone in clones:
                for s, t, vals in all_grns[method][clone].edges(data=True):
                    G_data[method][clone]["source"].append(s)
                    G_data[method][clone]["target"].append(t)
                    try:
                        G_data[method][clone]["support_0.90"].append(vals["support_0.9"])
                    except KeyError:
                        G_data[method][clone]["support_0.90"].append(0)

        G_df = {method:{clone:pd.DataFrame(data=G_data[method][clone]) for clone in clones} for method in methods+['scenic']}

        for method in methods+['scenic']:
            cl_edges = {}
            for clone in clones:
                unique_edges = set(all_grns[method][clone].edges())
                for clone2 in clones:
                    if clone2 != clone:
                        unique_edges = unique_edges - set(all_grns[method][clone2].edges())
                cl_edges[clone] = unique_edges
                G_df[method][clone]["edge"] = list(zip(G_df[method][clone]["source"], G_df[method][clone]["target"]))
                G_df[method][clone]['unique'] = [True if edge in cl_edges[clone] else False for edge in G_df[method][clone].edge]

        total_G_df[dataset] = G_df.copy()
        total_all_grns[dataset] = all_grns.copy()

pkl.dump(total_G_df, open(DATA_ROOT + '/TNBC_total_G_df_new.pkl','wb'))
pkl.dump(total_all_grns, open(DATA_ROOT + '/TNBC_total_all_grns_new.pkl','wb'))

In [ ]:
total_G_df = pkl.load(open(DATA_ROOT + '/Glio_total_G_df_new.pkl','rb'))
total_all_grns = pkl.load(open(DATA_ROOT + '/Glio_total_all_grns_new.pkl','rb'))
modules = pd.read_csv(DATA_ROOT + '/Glio/Glio_meta_modules.csv').to_dict(orient='list')

In [ ]:
total_all_grns['CID44991']['scenic']['0']

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import r2_score
from collections import defaultdict


def find_pairwise_row_outliers(df, n_std=2):

    outliers = {}

    # all pairwise row combinations
    for r1, r2 in combinations(df.index, 2):

        X = df.loc[r1].values
        Y = df.loc[r2].values

        # fit straight line Y = mX + c
        m, c = np.polyfit(X, Y, 1)
        Y_pred = m * X + c

        residuals = Y - Y_pred
        std_res = np.std(residuals)

        pair_key = str(r1)+'::'+str(r2)
        outliers[pair_key] = {'R2':r2_score(X, Y), 'outliers':[]}

        for i, col in enumerate(df.columns):
            if abs(residuals[i]) > n_std * std_res:
                outliers[pair_key]['outliers'].append({
                    "column": col,
                    "point": (Y[i], X[i]),
                    "residual": residuals[i]
                })

    return outliers

def find_sample_outlier_nodes(graph_dict, centrality='degree', thresh=2.0):
    """
    Detect nodes with outlier centrality values across group networks within each sample.

    Parameters
    ----------
    graph_dict : dict
        {'sample': {'group': nx_graph}}
    centrality : str
        'degree', 'betweenness', 'closeness', or 'pagerank'
    z_thresh : float
        Z-score threshold to define outliers

    Returns
    -------
    dict
        {'sample': {'group': [outlier_nodes]}}
    """

    centrality_funcs = {
        'degree': nx.degree_centrality,
        'betweenness': nx.betweenness_centrality,
        'closeness': nx.closeness_centrality,
        'pagerank': nx.pagerank
    }

    if centrality not in centrality_funcs:
        raise ValueError("Unsupported centrality metric")

    results = {}
    sample_df = {}
    sample_zscores = {}

    for sample, groups in graph_dict.items():

        # compute centrality for each group
        group_cent = {}
        all_nodes = set()

        for group, G in groups.items():
            cent = centrality_funcs[centrality](G)
            group_cent[group] = cent
            all_nodes.update(cent.keys())

        all_nodes = sorted(all_nodes)

        # build centrality matrix
        data = []
        for group in group_cent:
            row = [group_cent[group].get(n, 0) for n in all_nodes]
            data.append(row)

        df = pd.DataFrame(data, index=groups.keys(), columns=all_nodes)

        # compute z-scores per node across groups
        #zscores = (df - df.mean()) / df.std(ddof=0)
        #zscores = zscores.fillna(0)

        #sample_outliers = {}

        #for group in zscores.index:
        #    out_nodes = zscores.columns[np.abs(zscores.loc[group]) > z_thresh]
        #    sample_outliers[group] = list(out_nodes)

        results[sample] = find_pairwise_row_outliers(df, n_std=thresh)
        sample_df[sample] = df.copy()
        #sample_zscores[sample] = zscores.copy()

    return sample_df, sample_zscores, results


def build_outlier_matrix(result_dict):
    """
    result_dict format:
    dict[sample][pair] = {'R2': n1, 'outliers': [d1, d2, ...]}
    where di = {'column': val1, 'point': val2, 'residual': val3}
    """

    counts = defaultdict(lambda: defaultdict(int))
    all_columns = set()

    for sample, pairs in result_dict.items():
        for pair, pair_data in pairs.items():
            if 'N' in pair:
                continue
            for d in pair_data.get('outliers', []):
                col = d['column']
                counts[sample][col] += (1/len([x for x in pairs.keys() if 'N' not in x]))
                all_columns.add(col)

    all_columns = sorted(all_columns)

    # build dataframe
    df = pd.DataFrame(0, index=result_dict.keys(), columns=all_columns)

    for sample in counts:
        for col, c in counts[sample].items():
            df.loc[sample, col] = c

    return df

In [ ]:
centralities = {}
for centrality in ['degree', 'betweenness']:
    print(centrality)
    centralities[centrality] = {}
    centralities[centrality]['centrality_df'], centralities[centrality]['zscore_df'], centralities[centrality]['outliers'] = find_sample_outlier_nodes({k:v['scenic'] for k,v in total_all_grns.items()}, centrality=centrality, thresh=3)
    centralities[centrality]['matrix'] = build_outlier_matrix(centralities[centrality]['outliers'])

In [ ]:
for centrality in ['degree', 'betweenness']:
    for module, genes in modules.items():
        if module in scenic_module_bootstrap_adata.keys():
            if len(centralities[centrality]['matrix'].T.loc[centralities[centrality]['matrix'].T.index.isin(genes)].index) > 0:
                plt.figure(figsize=(9, 9))
                sns.heatmap(centralities[centrality]['matrix'].T.loc[centralities[centrality]['matrix'].T.index.isin(genes)], annot=True, square=True, cmap='Blues')
                plt.title(centrality+' - '+module)
                plt.show()

In [ ]:
for centrality in ['degree', 'betweenness', 'closeness', 'pagerank']:
    for module, genes in {'clone level':[k for k in gene_hits.keys()],'sample level':[k for k in gene_sample_hits.keys()],'sample multi level':[k for k in gene_multi_sample_hits.keys()]}.items():
        if len(centralities[centrality]['matrix'].T.loc[centralities[centrality]['matrix'].T.index.isin(genes)].index) > 0:
            plt.figure(figsize=(5, 5))
            sns.heatmap(centralities[centrality]['matrix'].T.loc[centralities[centrality]['matrix'].T.index.isin(genes)], annot=True, square=True, cmap='Blues')
            plt.title(centrality+' - '+module)
            plt.show()

In [ ]:
{centralities['betweenness']['outliers']['CID4495'][x]['R2'] for x in centralities['betweenness']['outliers']['CID4495'].keys()}

In [ ]:
from sklearn.metrics import r2_score

cent = 'betweenness'
samp = 'MGH105'

plt.scatter(centralities[cent]['centrality_df'][samp].T[['1']], centralities[cent]['centrality_df'][samp].T[['2']])
plt.title('R2 = '+str(r2_score(centralities[cent]['centrality_df'][samp].T[['1']], centralities[cent]['centrality_df'][samp].T[['2']])))

In [ ]:
zscores_df['CID4495']

In [ ]:
pathways = pd.read_csv(DATA_ROOT + '/GBM_relevant_pathways.csv')
top_pathways = list(pd.read_csv(DATA_ROOT + '/GBM_top30_pathways.csv')['Pathway'])
clone_pathways = dict(zip(pathways['clone_pathway'], pathways['clone_score']))
sample_pathways = dict(zip(pathways['sample_pathway'], pathways['sample_score']))
multi_sample_pathways = dict(zip(pathways['multi_sample_pathway'], pathways['multi_sample_score']))

all_pathways = {'clone':{k:v for k,v in clone_pathways.items() if k in top_pathways}, 'sample':{k:v for k,v in sample_pathways.items() if k in top_pathways}, 'multi sample':{k:v for k,v in multi_sample_pathways.items() if k in top_pathways}}

In [ ]:
TNBC_pathways = {'multi sample':{'PI3K-Akt signaling pathway':13.41109484, 'IL26 signaling':8.692055062, 'IL19 signaling':4.615835117, 'Apoptosis':4.461856683, 'TNF signaling pathway':2.509289591, 'IL-17 signaling pathway':2.456214493, 'Cytokine-cytokine receptor interaction':1.735431434, 'Interferon/MHC-II (II)':1.301403612, 'Hippo signaling pathway':9.601532, 'TP53 network':6.874999, 'FoxO signaling pathway':13.652324},
    'clone':{'PI3K-Akt signaling pathway':7.381299746, 'ErbB signaling pathway':4.759278464, 'EGF EGFR signaling':1.892102083, 'Targeted therapy in breast cancer':1.476180957, 'Hippo signaling pathway':2.494238, 'FoxO signaling pathway':6.074126},
    'sample':{'PI3K-Akt signaling pathway':5.687679164, 'Notch signaling pathway':5.251859462, 'TP53 network':1.394253, 'FoxO signaling pathway':4.095429}    
    }

Glio_pathways = {'multi sample':{'Neurogenesis regulation in the olfactory epithelium':14.4907826, 'TGF-beta signaling pathway':5.632316195, 'TP53 network':4.52481331, 'Glioma':3.664115434, 'IL26 signaling':2.6823492, 'Notch signaling pathway':2.627651532, 'Central carbon metabolism in cancer':1.313543384, 'NPC1':1.301921042, 'NPC2':1.301921042, 'Apoptosis':2.934517, 'PI3K-Akt signaling pathway':7.469099},
                'sample':{'B cell receptor signaling pathway':2.740998992, 'p53 transcriptional gene network':1.468326662, 'p53 signaling pathway':2.967579, 'Apoptosis':1.704994},
                'clone':{'TROP2 regulatory signaling':9.338475875, 'Ras signaling pathway':6.055690915, 'Glioma':4.574993036, 'Hallmark of cancer metastasis and epithelial to mesenchymal transition':4.299299902, 'Rap1 signaling pathway':3.985627848, 'EGFR tyrosine kinase inhibitor resistance':3.421432849, 'Wnt signaling pathway':2.319459885, 'PI3K-Akt signaling pathway':13.450101} 
                }

#for key in Glio_pathways:
#    sns.barplot(Glio_pathways[key], orient='y', color='#B4385E')
#    plt.xlabel('Pathway enrichment score')
#    plt.title(key)
#    plt.show()

In [ ]:
combined = {
    key: {**all_pathways.get(key, {}), **Glio_pathways.get(key, {})}
    for key in all_pathways.keys() | Glio_pathways.keys()
}

g= sns.clustermap(pd.DataFrame(combined).fillna(0), square=True, cmap='Reds', linewidths=0.6, linecolor='black', clip_on=False, vmin=0, col_cluster=False,  figsize=(1.5,16))
g.ax_row_dendrogram.set_visible(False)

In [ ]:
pd.DataFrame(combined).fillna(0)

In [ ]:
sns.heatmap(pd.DataFrame(Glio_pathways), square=True, cmap='Reds', linewidths=0.6, linecolor='black', clip_on=False, vmin=0)

# Patient survival

In [ ]:

# -----------------------------
# Load TCGA data
# -----------------------------
from tqdm import tqdm

'''
def load_tcga_xena(cohort="TCGA-GBM"):

    toil_hub = "https://toil.xenahubs.net"
    tcga_hub = "https://tcga.xenahubs.net"

    expr_dataset = "tcga_RSEM_Hugo_norm_count"

    cohort_map = {
        "TCGA-GBM":  "TCGA.GBM.sampleMap/GBM_clinicalMatrix",
        "TCGA-BRCA": "TCGA.BRCA.sampleMap/BRCA_clinicalMatrix",
    }
    phen_dataset = cohort_map[cohort]

    # Get samples from clinical dataset
    samples = xena.dataset_samples(tcga_hub, phen_dataset, None)
    print(f"Clinical samples: {len(samples)}")

    # Intersect with toil expression samples
    expr_samples = xena.dataset_samples(toil_hub, expr_dataset, None)
    samples = list(set(samples) & set(expr_samples))
    print(f"Samples with expression data: {len(samples)}")

    # Get gene list
    genes = xena.dataset_field(toil_hub, expr_dataset)

    # dataset_probe_values returns (positions, {gene: [one_value_per_sample]})
    # The values list length == len(samples) passed in, but we must build
    # the DataFrame as genes x samples, then transpose
    _, expr_dict = xena.dataset_probe_values(toil_hub, expr_dataset, samples, genes)

    # expr_dict is {gene: [values for each sample]} — build as genes x samples, then T
    expr = pd.DataFrame(expr_dict, index=genes, columns=samples).T
    # Now shape is (genes x samples) — transpose to (samples x genes)
    #expr = expr.T  # final shape: samples x genes

    print(f"Expression matrix shape: {expr.shape}")

    # Fetch clinical data field by field
    phen_fields = xena.dataset_field(tcga_hub, phen_dataset)

    clin_dict = {}
    for field in phen_fields:
        values = xena.dataset_fetch(tcga_hub, phen_dataset, samples, [field])
        clin_dict[field] = values[0] if values else [None] * len(samples)

    clin = pd.DataFrame(clin_dict, index=samples)
    print(f"Clinical matrix shape: {clin.shape}")

    return expr, clin
'''

def load_tcga_xena(cohort="TCGA-GBM"):

    toil_hub = "https://toil.xenahubs.net"
    tcga_hub = "https://tcga.xenahubs.net"

    expr_dataset = "tcga_RSEM_Hugo_norm_count"
    cohort_map = {
        "TCGA-GBM":  "TCGA.GBM.sampleMap/GBM_clinicalMatrix",
        "TCGA-BRCA": "TCGA.BRCA.sampleMap/BRCA_clinicalMatrix",
    }
    phen_dataset = cohort_map[cohort]

    # Get samples
    clin_samples = xena.dataset_samples(tcga_hub, phen_dataset, None)
    expr_samples = xena.dataset_samples(toil_hub, expr_dataset, None)

    print(f"Clinical samples: {len(clin_samples)}")
    print(f"Toil expression samples: {len(expr_samples)}")

    # Match on first 15 chars (TCGA-XX-XXXX-XX) to align barcode formats
    clin_map = {s[:15]: s for s in clin_samples}
    expr_map = {s[:15]: s for s in expr_samples}

    common = set(clin_map.keys()) & set(expr_map.keys())
    print(f"Matched samples: {len(common)}")

    clin_samples_matched = [clin_map[k] for k in common]
    expr_samples_matched = [expr_map[k] for k in common]

    # Get gene list
    genes = xena.dataset_field(toil_hub, expr_dataset)

    # Fetch expression using toil sample IDs
    _, expr_dict = xena.dataset_probe_values(toil_hub, expr_dataset, expr_samples_matched, genes)
    expr = pd.DataFrame(expr_dict, index=genes, columns=expr_samples_matched).T
    expr.index = clin_samples_matched  # realign index to clinical IDs
    print(f"Expression matrix shape: {expr.shape}")

    # Fetch clinical data using clinical sample IDs
    phen_fields = xena.dataset_field(tcga_hub, phen_dataset)
    clin_dict = {}
    for field in phen_fields:
        values = xena.dataset_fetch(tcga_hub, phen_dataset, clin_samples_matched, [field])
        clin_dict[field] = values[0] if values else [None] * len(clin_samples_matched)

    clin = pd.DataFrame(clin_dict, index=clin_samples_matched)
    print(f"Clinical matrix shape: {clin.shape}")

    return expr, clin

# -----------------------------
# Main function
# -----------------------------
def program_survival_analysis(
    adata,
    expr,
    clin,
    program_col,
    cancer_type="TNBC",   # "TNBC" or "GBM"
    split="median",
    figw=6,
    figh=6,
    fontsize=10
):
    """
    Test if gene program enrichment is associated with survival.

    Parameters
    ----------
    adata : AnnData
        Must contain program scores in .obs
    program_col : str
        Column in adata.obs with gene program scores
    cancer_type : str
        "TNBC" or "GBM"
    split : str
        "median" or "upper_quartile"
    """

    # -----------------------------
    # 1. Load TCGA
    # -----------------------------
    if cancer_type == "TNBC":
        # TNBC filter
        clin = clin[
            (clin["breast_carcinoma_estrogen_receptor_status"] == 1) &   # Negative
            (clin["breast_carcinoma_progesterone_receptor_status"] == 1) & # Negative
            (clin["lab_proc_her2_neu_immunohistochemistry_receptor_status"] == 0)  # Negative
        ]

    # -----------------------------
    # 2. Align samples
    # -----------------------------
    prog = adata.to_df()[[program_col]].copy()

    common = prog.index.intersection(clin.index)
    prog = prog.loc[common]
    clin = clin.loc[common]


    # -----------------------------
    # 3. Survival preprocessing
    # -----------------------------
    clin["time"] = clin["days_to_death"].astype(str).astype(float).fillna(clin["days_to_last_followup"].astype(str).astype(float))
    if cancer_type == "TNBC":
        clin["event"] = (clin["vital_status"] == 1).astype(int)
    if cancer_type == "GBM":
        clin["event"] = (clin["vital_status"] == 2).astype(int)

    df = pd.concat([clin[["time", "event"]], prog], axis=1).dropna()

    # -----------------------------
    # 4. Define enrichment groups
    # -----------------------------
    scores = df[program_col]

    if split == "median":
        cutoff = scores.median()
    elif split == "upper_quartile":
        cutoff = scores.quantile(0.75)
    elif split == "lower_quartile":
        cutoff = scores.quantile(0.25)
    else:
        raise ValueError("Invalid split")

    df["group"] = scores >= cutoff
    df = df.dropna(subset=['time', 'event'])
    print(df)
    print(df['event'].unique())

    # -----------------------------
    # 5. Kaplan-Meier
    # -----------------------------
    plt.rcParams.update({'font.size': fontsize})
    
    kmf = KaplanMeierFitter()

    high = df[df["group"]]
    low = df[~df["group"]]

    kmf.fit(high["time"], high["event"], label="High expression")
    kmf.plot()

    kmf.fit(low["time"], low["event"], label="Low expression")
    kmf.plot()

    # log-rank test
    lr = logrank_test(
        high["time"], low["time"],
        event_observed_A=high["event"],
        event_observed_B=low["event"]
    )
    
    
    plt.legend(fontsize=fontsize-3)
    plt.gcf().set_size_inches(figw, figh)
    plt.title(f"{program_col} ({cancer_type}) p={lr.p_value:.3e}")
    plt.xlabel("Days")
    plt.ylabel("Survival probability")
    plt.show()


    # -----------------------------
    # 6. Cox model
    # -----------------------------
    cph = CoxPHFitter()
    cph.fit(df[["time", "event", program_col]], duration_col="time", event_col="event")

    cox_summary = cph.summary


    return {
        "logrank_p": lr.p_value,
        "cox_summary": cox_summary,
        "n_patients": df.shape[0]
    }
    
def program_survival_analysis_all_genes(
    adata,
    expr,
    clin,
    cancer_type="TNBC",
    split="median"
):
    results = []

    # Iterate over all columns in adata (each is a candidate gene/program)
    candidate_cols = adata.to_df().columns.tolist()

    for program_col in tqdm(candidate_cols):
        try:
            # -----------------------------
            # 1. Filter clinical data
            # -----------------------------
            clin_sub = clin.copy()
            if cancer_type == "TNBC":
                clin_sub = clin_sub[
                    (clin_sub["breast_carcinoma_estrogen_receptor_status"] == 1) &
                    (clin_sub["breast_carcinoma_progesterone_receptor_status"] == 1) &
                    (clin_sub["lab_proc_her2_neu_immunohistochemistry_receptor_status"] == 0)
                ]

            # -----------------------------
            # 2. Align samples
            # -----------------------------
            prog = adata.to_df()[[program_col]].copy()
            common = prog.index.intersection(clin_sub.index)
            prog = prog.loc[common]
            clin_sub = clin_sub.loc[common]

            # -----------------------------
            # 3. Survival preprocessing
            # -----------------------------
            clin_sub["time"] = (
                clin_sub["days_to_death"]
                .astype(str).astype(float)
                .fillna(clin_sub["days_to_last_followup"].astype(str).astype(float))
            )
            if cancer_type == "TNBC":
                clin_sub["event"] = (clin_sub["vital_status"] == 1).astype(int)
            elif cancer_type == "GBM":
                clin_sub["event"] = (clin_sub["vital_status"] == 2).astype(int)

            df = pd.concat([clin_sub[["time", "event"]], prog], axis=1).dropna()

            if df.shape[0] < 10:  # skip if too few patients
                continue

            # -----------------------------
            # 4. Define enrichment groups
            # -----------------------------
            scores = df[program_col]
            if split == "median":
                cutoff = scores.median()
            elif split == "upper_quartile":
                cutoff = scores.quantile(0.75)
            elif split == "lower_quartile":
                cutoff = scores.quantile(0.25)
            else:
                raise ValueError("Invalid split")

            df["group"] = scores >= cutoff
            df = df.dropna(subset=["time", "event"])

            high = df[df["group"]]
            low  = df[~df["group"]]

            if len(high) < 2 or len(low) < 2:  # skip degenerate splits
                continue

            # -----------------------------
            # 5. Log-rank test (no plotting)
            # -----------------------------
            lr = logrank_test(
                high["time"], low["time"],
                event_observed_A=high["event"],
                event_observed_B=low["event"]
            )

            # -----------------------------
            # 6. Collect if significant
            # -----------------------------
            if lr.p_value < 0.05:
                results.append({
                    "gene": program_col,
                    "logrank_p": lr.p_value,
                    "n_patients": df.shape[0],
                    "n_high": len(high),
                    "n_low": len(low),
                })

        except Exception as e:
            print(f"Skipping {program_col}: {e}")
            continue

    # Return as a DataFrame sorted by p-value
    results_df = pd.DataFrame(results).sort_values("logrank_p").reset_index(drop=True)
    return results_df

In [ ]:
sampletype = 'TNBC'
expr, clin = load_tcga_xena("TCGA-BRCA")

In [ ]:
adata = sc.AnnData(expr)
dc.pl.filter_by_expr(
    adata=adata,
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.5,
)
dc.pl.filter_by_expr(
    adata=adata,
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.5,
)

In [ ]:
# Store raw counts in layers
adata.layers["counts"] = adata.X.copy()

# Normalize, scale and compute pca
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata)

# Return raw counts to X
dc.pp.swap_layer(adata=adata, key="counts", inplace=True)

In [ ]:
'''
programs = {
    "TNBC": {
        "clone": [
            "ZKSCAN7", "GRH2", "NONO", "MYBL1", "SIX1", "FOXJ1",
            "PRDM2", "HEY1", "HEY2", "MED1", "PBRM1"
        ],
        "sample": [
            "SOX10", "ID4", "TGIF1", "DLX5", "NR2E1"
        ],
        "multi": [
            "FOXD3", "MXI1", "HOXB13", "TVF7", "BHLHE40", "TRPS1",
            "ZNF143", "LMX1B", "SP7", "KLF11", "MAFG", "NRI1D1",
            "NFIX", "TCF21", "KLF13", "FOXQ1", "PROX1", "TEAD4",
            "ETV1", "MAFB", "EHF", "DLX2", "BCL11A", "TBX3",
            "PLAGL1", "PATZ1", "POU2F3", "NSD2", "CTBP2", "TBX5",
            "POU3F1", "E2F2", "ZNF384"
        ]
    },
    "GBM": {
        "clone": [
            "NR2F2", "LMO2", "NEUROD2", "BCL11B", "EMX2", "ZNF354C",
            "HOXB7", "KCNIP3"
        ],
        "sample": [
            "PKNOX1", "EBF1", "CEBPZ", "PEG3"
        ],
        "multi": [
            "DLX2", "WWTR1", "BCL11A", "SMARCC1", "DLX1", "ZFPM2",
            "NONO", "SOX11", "MSC", "MNT", "NEUROG1", "HMGB2",
            "HEY1", "MYT1", "FOXP1"
        ]
    }
}
'''

programs = {
    "TNBC": {
        "interferon_outlier": [
            'KLF4', 'KLF10', 'MAFF'
        ],
        "interferon_tf": [
            'KLF4', 'KLF10', 'MAFF', 'CEBPD', 'ARID5B'
        ],
        "adherens_outlier": [
            'CREB3L2'
        ],
        "pdac_tf": [
            'JUND', 'GRHL1'
        ],
        "pdac_tf": [
            'JUND', 'GRHL1'
        ],
        "pdac_outlier": [
            'JUND'
        ],
        "translation_tf": [
            'MYC', 'HES4'
        ],
        "stress_tf": [
            'DDIT3', 'AHR', 'ATF4'
        ],
    },
    "GBM": {
        "npc1_tf": [
            'ASCL1', 'TCF12', 'OLIG1', 'SOX11', 'SOX4', 'ETV1'
        ],
        "npc1_outlier": [
            'ETV1', 'SOX11', 'TCF12'
        ],
        "npc2_tf": [
            'TCF4', 'NFIB', 'NFIX', 'SOX11', 'SOX4'
        ],
        "npc2_outlier": [
            'TCF4', 'NFIB', 'NFIX', 'SOX4'
        ],
        "npc2_outlier": [
            'TCF4', 'NFIB', 'NFIX', 'SOX4'
        ],
        "ac_tf": [
            'HEY1'
        ],
        "mes2_tf": [
            'DDIT3', 'EPAS1', 'ATF3', ''
        ],
        "stress_tf": [
            'ATF4', 'KLF10', 'DDIT3'
        ],
        "interferon_tf": [
            'IRF9', 'STAT1'
        ],
    }
}

pathway = dc.op.hallmark(organism="human")
#Add meta modules
pathway = pathway.to_dict(orient='list')
for module, related_genes in programs[sampletype].items():
    for gene in related_genes:
        pathway['source'].append(sampletype+'_'+module)
        pathway['target'].append(gene)
pathway = pd.DataFrame(pathway)
dc.mt.ulm(data=adata, net=pathway, tmin=1)
pathway_scores = dc.pp.get_obsm(adata=adata, key="score_ulm")

In [ ]:
stats = program_survival_analysis(adata=adata, expr=expr, clin=clin, program_col='KLF10', cancer_type=sampletype, split="median", figh=9, figw=9, fontsize=20)
#survival_genes_df = program_survival_analysis_all_genes(adata=adata, expr=expr, clin=clin, cancer_type=sampletype, split="median")
'''
df = []
for program in programs[sampletype].keys():
    stats = program_survival_analysis(adata=pathway_scores, expr=expr, clin=clin, program_col=sampletype+'_'+program, cancer_type=sampletype, split="median")
    df.append({'program':program, 'pval':stats['logrank_p']})
pd.DataFrame(df).to_csv(DATA_ROOT + '/TNBC_survival_programs.csv', index=False)
'''

In [ ]:
survival_genes_df = pd.read_csv(DATA_ROOT + '/Glio_survival_genes.csv')
modules = pd.read_csv(DATA_ROOT + '/Glio/Glio_meta_modules.csv').to_dict(orient='list')
intersections = {k:set(v).intersection(set(survival_genes_df['gene'].tolist())) for k,v in modules.items()}
print(intersections)

In [ ]:
tcga_hub = "https://tcga.xenahubs.net"

clin_samples = xena.dataset_samples(tcga_hub, "TCGA.GBM.sampleMap/GBM_clinicalMatrix", None)
expr_samples = xena.dataset_samples(tcga_hub, "TCGA.GBM.sampleMap/HiSeqV2", None)

print(f"Clinical samples: {len(clin_samples)}")
print(f"Expression samples: {len(expr_samples)}")
print(f"Intersection: {len(set(clin_samples) & set(expr_samples))}")

# Check if ID formats match
print("\nClinical sample IDs (first 3):", clin_samples[:3])
print("Expression sample IDs (first 3):", expr_samples[:3])

In [ ]:
# Match on patient barcode prefix (first 12 chars)
clin_map = {s[:12]: s for s in clin_samples}
expr_map = {s[:12]: s for s in expr_samples}

common_patients = set(clin_map.keys()) & set(expr_map.keys())
print(f"Matched patients: {len(common_patients)}")

samples = [expr_map[p] for p in common_patients]